# RAISE — combined experiment notebook

Everything needed to reproduce the experiments behind **"RAISE: Representation Agnostic Information and Separation Estimation"**,
in the order the paper uses them. Each section header says where the code came from:

* **VERBATIM** – copied unchanged from one of your uploaded notebooks (name + cell index given).
* **PATCHED** – copied, with a small mechanical change (a `QUICK` switch or a path); the patch is listed.
* **FIXED** – your notebook existed but was broken / unfinished; rewritten here.
* **GENERATED** – no notebook in the zip contained this experiment; written from the paper's description. Assumptions are listed in the cell.

`QUICK = True` (next cell) runs everything at toy scale to check that the pipeline executes. Set `QUICK = False` for paper-grade numbers.
Sections 8–9 need network access (NASA Earthdata login, USGS 3DEP, Copernicus DEM) and a GPU for the neural field; run them on Colab.

In [ ]:
# Colab / fresh environment. (laspy+lazrs, pyproj, h5py, rasterio only needed for sections 8-9.)
# !pip install -q numpy scipy pandas matplotlib scikit-learn statsmodels torch laspy[lazrs] pyproj h5py rasterio requests

In [ ]:
# ---------------- RAISE combined notebook: global config ----------------
# QUICK=True  -> minutes-scale smoke test (reduced Monte Carlo / epochs). Numbers will NOT match the paper.
# QUICK=False -> paper-grade ("locked") settings. Expect hours on Colab for sections 4, 6, 9.
QUICK = True

import os, json, time, pickle, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

OUT = "raise_outputs"; os.makedirs(OUT, exist_ok=True)
SEED = 0
rng_global = np.random.default_rng(SEED)
_trapz = getattr(np, "trapezoid", None) or np.trapz

KAPPA = 2*np.sqrt(np.log(2))          # = 1.6651, paper Sec.3 footnote (FWHM convention)  -> H_NWCS = KAPPA*sigma_pulse
SIGMA_PULSE_GEDI_M = 1.3503           # paper: sigma_pulse for GEDI's 15 ns pulse (one-way metres)

# gates for the heavy copied cells (App. E sweeps are only needed for the full paper-grade run)
RUN_APPE_SWEEPS = not QUICK
print("QUICK =", QUICK)

In [ ]:
# ---------------- Shared fast helpers (vectorised; used by new sections) ----------------
def P_(t, tau, s):  return np.exp(-(t - tau)**2 / (2*s**2))
def dP_(t, tau, s): return (t - tau)/s**2 * P_(t, tau, s)          # d/dtau

def mu_cd(c, d, t, s, w1, w2):
    """mean waveform(s). c,d may be arrays of shape (B,) -> returns (B, len(t))."""
    c = np.atleast_1d(c)[:, None]; d = np.atleast_1d(d)[:, None]
    return w1*P_(t[None, :], c - d/2, s) + w2*P_(t[None, :], c + d/2, s)

def FD_exact_fast(d, t, s, sn, w1, w2, crange, nc, nmc, rng, chunk=2000):
    """Exact marginal Fisher information F_D(d): centroid c integrated out (uniform prior on crange).
       Same estimator as the original 'Contribution 4' script, vectorised over waveforms with matrix products.
       F_D = E_x[ ( E_{c|x,d}[ local score ] )^2 ],  local score = (x - mu_k).dmu_k / sn^2."""
    cg = np.linspace(crange[0], crange[1], nc)
    t1, t2 = cg - d/2, cg + d/2
    MU  = w1*P_(t[None,:], t1[:,None], s) + w2*P_(t[None,:], t2[:,None], s)            # (nc, nt)
    DMU = -(w1/2)*dP_(t[None,:], t1[:,None], s) + (w2/2)*dP_(t[None,:], t2[:,None], s)   # dmu/dd
    mm = (MU*MU).sum(1); md = (MU*DMU).sum(1)
    s2 = sn**2
    sq = []
    done = 0
    while done < nmc:
        B = min(chunk, nmc - done)
        ctrue = rng.uniform(crange[0], crange[1], B)
        X = mu_cd(ctrue, np.full(B, d), t, s, w1, w2) + rng.normal(0, sn, (B, len(t)))
        XM = X @ MU.T                                                                     # (B, nc)
        ll = -0.5*((X*X).sum(1)[:, None] - 2*XM + mm[None, :]) / s2
        ll -= ll.max(1, keepdims=True)
        w = np.exp(ll); w /= w.sum(1, keepdims=True)
        score = (X @ DMU.T - md[None, :]) / s2
        sq.append(((w*score).sum(1))**2)
        done += B
    sq = np.concatenate(sq)
    return sq.mean(), sq.std(ddof=1)/np.sqrt(len(sq))

def schur_info(d, t, s, sn, w1, w2, c=0.0):
    """Profile-likelihood / Schur-complement Fisher information  I_dd - I_dc^2 / I_cc  at fixed centroid c."""
    t1, t2 = c - d/2, c + d/2
    dmu_dd = -(w1/2)*dP_(t, t1, s) + (w2/2)*dP_(t, t2, s)
    dmu_dc =  w1*dP_(t, t1, s) +  w2*dP_(t, t2, s)
    s2 = sn**2
    Idd = (dmu_dd**2).sum()/s2; Idc = (dmu_dd*dmu_dc).sum()/s2; Icc = (dmu_dc**2).sum()/s2
    return Idd - Idc**2/Icc

## 1. Exact marginal Fisher information → mutual-information ceiling  (Sec. 2, Eq. 4–7, Fig. 1c; headline **2.00 nats**)
*Source:* `Untitled4 (1).ipynb` cell 1 — **PATCHED** (`PRODUCTION = True` → `PRODUCTION = not QUICK`).
Config (σ_pulse=1, σ_n=0.2, w=(0.4,0.6), c~U[−1,1], d∈[0.001,3]). Your logged result: 2.00244 nats, MC-propagated range [1.993, 2.012].

**Read before quoting:** this config is *not* the GEDI config (σ_pulse=1.3503 m). The 2.00 nats is a property of this synthetic sensor, and it depends on the
prior range of d (the bound integrates √F_D over [0, 3]). The paper's sentence "for the sensor configuration studied here" is correct but the sensor is the toy one.

In [ ]:
"""
=====================================================================
Contribution 4 closure: exact marginal Fisher information -> Goerecki
MI bound, for the waveform centroid/separation nuisance model.
=====================================================================

WHAT THIS DOES
--------------
Computes I(X; D) <= ln(1 + 0.5 * integral sqrt(F_D_marginal(d)) dd)
directly from the TRUE marginal Fisher information F_D_marginal(d),
where the centroid c is genuinely marginalized (integrated against its
prior), not profiled/Schur-complemented. No J_eff, no approximation
ambiguity -- Goerecki et al.'s Theorem 1 is applied to exactly the
object it is stated for.

Self-contained: paste this whole file and run. Only dependency is numpy.

HOW TO TUNE FOR SPEED VS. PRECISION
------------------------------------
Set PRODUCTION = True for paper-quality numbers (slow: minutes).
Set PRODUCTION = False for a quick correctness check (seconds).
The two knobs that matter:
  - N_C_GRID   : inner quadrature resolution for the posterior over c.
                 Convergence is O(1/N_C_GRID); at N_C_GRID=800 you are
                 within ~0.1% of the converged value (verified via
                 grid_convergence_check below -- re-verify if you
                 change sigma_pulse, w1, w2, or c_range).
  - N_MC_BASE / N_MC_BOOST : outer Monte Carlo sample counts. BOOST is
                 used near d=0 (the identifiability-collapse region)
                 where estimator variance is largest -- do not use a
                 flat n_mc across the whole sweep, check the printed
                 per-d relative SE to confirm boosting is enough for
                 YOUR model (the threshold scale is model-dependent).

CAVEATS (do not silently drop these when copying into the paper)
------------------------------------------------------------------
1. Score-marginalization identity d/dd ln p(x|d) = E_{c|x,d}[local score]
   requires the prior p(c) to NOT depend on d. Uniform prior on
   C_RANGE, fixed, satisfies this. If your actual centroid prior
   depends on separation, this needs Clarkson (2019) Sec. 3's extra
   term -- do not reuse this code unmodified in that case.
2. Two independent numerical-error axes (MC sampling vs. c-grid
   quadrature) are reported separately; MC_SE alone does not capture
   grid error -- see grid_convergence_check().
3. This produces a bound on I(X;D) for D supported on [D_MIN, D_MAX]
   (Goerecki Theorem 1, bounded-support case). If your prior on d has
   unbounded support, use Theorem 2 instead (differentiable p(d) over
   the whole real line) -- not implemented here, ask if you need it.
"""

import numpy as np
from dataclasses import dataclass

_trapz = getattr(np, "trapezoid", None) or np.trapz  # numpy>=2.0 renamed trapz


# =====================================================================
# 1. CONFIGURATION -- edit these to match your actual NWCS model
# =====================================================================

PRODUCTION = (not QUICK)   # <-- flip to True for paper-quality numbers (slower)

@dataclass
class ModelConfig:
    t_grid: np.ndarray       # sample times
    sigma_pulse: float       # pulse width
    sigma_n: float            # noise std per sample
    w1: float                 # pulse-1 weight
    w2: float                 # pulse-2 weight
    c_range: tuple             # (c_min, c_max) support of uniform prior on centroid


CFG = ModelConfig(
    t_grid=np.linspace(-5, 5, 100),
    sigma_pulse=1.0,
    sigma_n=0.2,
    w1=0.4,
    w2=0.6,
    c_range=(-1.0, 1.0),
)

# Prior support [D_MIN, D_MAX] for the separation parameter d
D_MIN, D_MAX = 0.001, 3.0

if PRODUCTION:
    N_C_GRID     = 800
    N_MC_BASE    = 4000
    N_MC_BOOST   = 12000
    D_THRESHOLD  = 0.5      # below this |d|, use N_MC_BOOST (tune to your model)
    D_GRID = np.concatenate([
        np.linspace(D_MIN, 0.5, 20),   # dense near identifiability collapse
        np.linspace(0.5, D_MAX, 15),
    ])
else:
    N_C_GRID     = 150
    N_MC_BASE    = 800
    N_MC_BOOST   = 2500
    D_THRESHOLD  = 0.5
    D_GRID = np.concatenate([
        np.linspace(D_MIN, 0.5, 6),
        np.linspace(0.5, D_MAX, 6),
    ])


# =====================================================================
# 2. MODEL: two-pulse Gaussian waveform, additive Gaussian noise
# =====================================================================

def pulse(t, tau, sigma_pulse):
    return np.exp(-(t - tau) ** 2 / (2 * sigma_pulse ** 2))


def pulse_deriv(t, tau, sigma_pulse):
    """d/dtau of pulse(t, tau, sigma_pulse)."""
    return (t - tau) / sigma_pulse ** 2 * pulse(t, tau, sigma_pulse)


def mu_waveform(c, d, cfg: ModelConfig):
    tau1, tau2 = c - d / 2.0, c + d / 2.0
    t = cfg.t_grid
    return cfg.w1 * pulse(t, tau1, cfg.sigma_pulse) + cfg.w2 * pulse(t, tau2, cfg.sigma_pulse)


def generate_noisy_waveform(c, d, cfg: ModelConfig, rng: np.random.Generator):
    return mu_waveform(c, d, cfg) + rng.normal(0.0, cfg.sigma_n, size=cfg.t_grid.shape)


def _posterior_and_scores_for_one_x(x_waveform, d, c_grid, cfg: ModelConfig):
    """
    Vectorized over the ENTIRE c_grid at once (no Python loop over c).
    Returns (posterior_weights, local_scores), both shape (n_c,).
    """
    t = cfg.t_grid[:, None]                    # (n_t, 1)
    c = c_grid[None, :]                        # (1, n_c)
    tau1, tau2 = c - d / 2.0, c + d / 2.0

    mu = cfg.w1 * pulse(t, tau1, cfg.sigma_pulse) + cfg.w2 * pulse(t, tau2, cfg.sigma_pulse)
    dmu_dd = -(cfg.w1 / 2.0) * pulse_deriv(t, tau1, cfg.sigma_pulse) \
             + (cfg.w2 / 2.0) * pulse_deriv(t, tau2, cfg.sigma_pulse)

    resid = x_waveform[:, None] - mu

    log_lik = -0.5 * np.sum(resid ** 2, axis=0) / cfg.sigma_n ** 2
    log_lik -= log_lik.max()
    w = np.exp(log_lik)
    w /= w.sum()

    local_scores = np.sum(resid * dmu_dd, axis=0) / cfg.sigma_n ** 2
    return w, local_scores


def marginal_score_d(x_waveform, d, c_grid, cfg: ModelConfig):
    """True marginal score d/dd ln p(x|d) = E_{c|x,d}[local score]."""
    w, local_scores = _posterior_and_scores_for_one_x(x_waveform, d, c_grid, cfg)
    return np.sum(w * local_scores)


# =====================================================================
# 3. MONTE CARLO ESTIMATOR of the TRUE marginal Fisher information
# =====================================================================

def F_D_marginal_montecarlo(d, cfg: ModelConfig, n_mc, n_c_grid, rng):
    c_grid = np.linspace(cfg.c_range[0], cfg.c_range[1], n_c_grid)
    c_true_samples = rng.uniform(cfg.c_range[0], cfg.c_range[1], size=n_mc)

    scores_sq = np.empty(n_mc)
    for i in range(n_mc):
        x = generate_noisy_waveform(c_true_samples[i], d, cfg, rng)
        s = marginal_score_d(x, d, c_grid, cfg)
        scores_sq[i] = s ** 2

    F_hat = scores_sq.mean()
    se_mc = scores_sq.std(ddof=1) / np.sqrt(n_mc)
    return F_hat, se_mc


def grid_convergence_check(d, cfg: ModelConfig, n_mc=800,
                            grid_sizes=(100, 200, 400, 800), seed=0):
    """Fixed MC stream, varying c_grid resolution -- isolates quadrature
    error from MC noise. Re-run this whenever you change the model."""
    print(f"\n--- Grid convergence check at d={d:.4f} (n_mc={n_mc} fixed) ---")
    for ng in grid_sizes:
        rng = np.random.default_rng(seed)
        F_hat, se = F_D_marginal_montecarlo(d, cfg, n_mc, ng, rng)
        print(f"  n_c_grid={ng:4d}   F_hat={F_hat:.6f}   MC_SE={se:.6f}")


def adaptive_n_mc(d, d_threshold, n_mc_base, n_mc_boost):
    return n_mc_boost if abs(d) < d_threshold else n_mc_base


def sweep_F_D_marginal(d_grid, cfg: ModelConfig, n_c_grid, d_threshold,
                        n_mc_base, n_mc_boost, seed=0):
    F_vals, SE_vals = [], []
    for i, d in enumerate(d_grid):
        n_mc = adaptive_n_mc(d, d_threshold, n_mc_base, n_mc_boost)
        rng = np.random.default_rng(seed + i)
        F_hat, se = F_D_marginal_montecarlo(d, cfg, n_mc, n_c_grid, rng)
        F_vals.append(F_hat)
        SE_vals.append(se)
        print(f"d={d:+.4f}   F_hat={F_hat:10.5f}   MC_SE={se:9.5f}   "
              f"rel_SE={se/max(F_hat,1e-12):.3%}   n_mc={n_mc}")
    return np.array(F_vals), np.array(SE_vals)


# =====================================================================
# 4. Goerecki et al. (2024/25) Theorem 1: MI bound from F(d)
# =====================================================================

def goerecki_MI_bound(d_grid, F_vals):
    """
    I(X; D) <= ln(1 + 0.5 * integral_a^b sqrt(F(d)) dd),
    valid for D supported on [a,b] = [d_grid[0], d_grid[-1]].
    """
    sqrt_F = np.sqrt(np.clip(F_vals, 0, None))
    integral = _trapz(sqrt_F, d_grid)
    bound = np.log(1.0 + 0.5 * integral)
    return bound, integral


def propagate_mc_error_to_bound(d_grid, F_vals, SE_vals):
    """Rough first-order MC-error propagation through the log-bound."""
    sqrt_F_hi = np.sqrt(np.clip(F_vals + SE_vals, 0, None))
    sqrt_F_lo = np.sqrt(np.clip(F_vals - SE_vals, 0, None))
    bound_hi = np.log(1.0 + 0.5 * _trapz(sqrt_F_hi, d_grid))
    bound_lo = np.log(1.0 + 0.5 * _trapz(sqrt_F_lo, d_grid))
    return bound_lo, bound_hi


# =====================================================================
# 5. RUN
# =====================================================================

if __name__ == "__main__":
    print(f"PRODUCTION mode: {PRODUCTION}")
    print(f"N_C_GRID={N_C_GRID}  N_MC_BASE={N_MC_BASE}  N_MC_BOOST={N_MC_BOOST}")

    # Step 0 -- re-verify quadrature convergence for THIS model config
    grid_convergence_check(d=0.3, cfg=CFG, n_mc=800,
                            grid_sizes=(100, 200, 400, N_C_GRID))

    # Step 1 -- exact marginal Fisher information across the d-prior support
    print("\n--- F_D_marginal(d) sweep (exact marginal, no J_eff) ---")
    F_vals, SE_vals = sweep_F_D_marginal(
        D_GRID, CFG, N_C_GRID, D_THRESHOLD, N_MC_BASE, N_MC_BOOST
    )

    # Step 2 -- Goerecki MI bound, applied directly (no substitution)
    bound, integral = goerecki_MI_bound(D_GRID, F_vals)
    bound_lo, bound_hi = propagate_mc_error_to_bound(D_GRID, F_vals, SE_vals)

    print(f"\nintegral_a^b sqrt(F_D_marginal(d)) dd  ~=  {integral:.5f}")
    print(f"Goerecki MI bound:  I(X; D) <= {bound:.5f} nats")
    print(f"MC-propagated bound range: [{bound_lo:.5f}, {bound_hi:.5f}] nats")
    print(f"\n(equivalently: I(X;D) <= {bound/np.log(2):.5f} bits, "
          f"range [{bound_lo/np.log(2):.5f}, {bound_hi/np.log(2):.5f}])")

In [ ]:
# ---------- Plot F_D(d) from the Section-1 run (the script above leaves D_GRID, F_vals, SE_vals in memory) ----------
try:
    plt.figure(figsize=(5, 3.4))
    plt.errorbar(D_GRID, F_vals, yerr=2*SE_vals, fmt="o-", ms=3, capsize=2)
    plt.axvline(KAPPA*CFG.sigma_pulse, ls=":", color="k", label="H_NWCS = kappa*sigma")
    plt.xlabel("separation d"); plt.ylabel("F_D(d)  (exact marginal)"); plt.legend(); plt.tight_layout()
    plt.savefig(f"{OUT}/sec1_FD_curve.png", dpi=200); plt.show()
    print("MI ceiling  I(X;D) <= %.4f nats   (paper: 2.00 nats)" % goerecki_MI_bound(D_GRID, F_vals)[0])
except NameError as e:
    print("Run the Section-1 cell first:", e)

### Figure 1 (regenerated)
*GENERATED.* The original Fig. 1 notebook was not in the zip. Fig. 1c in the paper peaks near F_D≈110 (GEDI-scale pulse); the noise level behind it is unknown, so `SN1` is a placeholder.

In [ ]:
# ---------------- Figure 1 (regenerated): waveform collapse + marginal Fisher collapse ----------------
# NOTE: the original Fig.1 notebook was not in the zip. Fig 1c in the paper peaks near F_D ~ 110 at d~3.5 with a
# GEDI-scale pulse (sigma=1.3503) -- the noise level used there is unknown. sn below is a placeholder; tune it to match.
S1, W1_, W2_ = SIGMA_PULSE_GEDI_M, 0.4, 0.6
SN1 = 0.15
H_NWCS_1 = KAPPA*S1                                   # = 2.249
t1 = np.linspace(-8, 8, 200)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
for d_, lab, col in [(5.0, "well separated (d=5.0)", "tab:blue"), (H_NWCS_1, f"near threshold (d={H_NWCS_1:.2f})", "tab:orange"), (0.8, "collapsed (d=0.8)", "tab:red")]:
    ax[0].plot(t1, mu_cd(0.0, d_, t1, S1, W1_, W2_)[0], color=col, label=lab)
ax[0].set_xlabel("time"); ax[0].set_ylabel("normalized return power"); ax[0].set_title("(b) waveform collapse"); ax[0].legend(fontsize=7)

dgrid = np.linspace(0.05, 4.0, 14 if QUICK else 30)
vals = [FD_exact_fast(d_, t1, S1, SN1, W1_, W2_, (-1.0, 1.0), 200 if QUICK else 800, 1000 if QUICK else 8000, rng_global) for d_ in dgrid]
F1 = np.array([v[0] for v in vals]); F1se = np.array([v[1] for v in vals])
ax[1].plot(dgrid, F1); ax[1].fill_between(dgrid, F1-2*F1se, F1+2*F1se, alpha=.25)
ax[1].axvline(H_NWCS_1, ls=":", color="k"); ax[1].set_xlabel("separation d"); ax[1].set_ylabel("F_D(d)"); ax[1].set_title("(c) Fisher collapse")
plt.tight_layout(); plt.savefig(f"{OUT}/fig1_regenerated.png", dpi=200); plt.show()
print("peak F_D = %.1f at d=%.2f   (paper Fig1c: ~110 at d~3.5)" % (F1.max(), dgrid[F1.argmax()]))

## 2. Schur-complement vs exact-marginal Fisher information  (Appendix C, Table 2)
*GENERATED.* No uploaded notebook computes this table. Same model/config as Section 1. The paper reports 73.7% divergence at d=0.05
(I_Schur=0.047, F_D=0.180, 4 seeds, n_c=800, n_mc=8000). The `(F_D − I_Schur)/F_D` definition of "divergence" is inferred from those two numbers.

In [ ]:
# ================= Experiment: Schur-complement vs exact-marginal Fisher information (Appendix C, Table 2) =================
# NOT present in any uploaded notebook -> generated. Config matches the Section-1 script (sigma=1, sn=0.2, w=(0.4,0.6), c~U[-1,1]).
T2 = dict(t=np.linspace(-5, 5, 100), s=1.0, sn=0.2, w1=0.4, w2=0.6, crange=(-1.0, 1.0))
d_list = [0.05, 0.10, 0.20, 0.30, 0.50, 1.0, 2.0, 3.0]
n_seeds = 4
nc_T2, nmc_T2 = 800, 8000        # paper settings; vectorised, so this takes ~10 s even in QUICK mode

rows = []
for d_ in d_list:
    Is = schur_info(d_, T2["t"], T2["s"], T2["sn"], T2["w1"], T2["w2"], c=0.0)
    Fs = [FD_exact_fast(d_, T2["t"], T2["s"], T2["sn"], T2["w1"], T2["w2"], T2["crange"], nc_T2, nmc_T2,
                        np.random.default_rng(1000*k + int(d_*100)))[0] for k in range(n_seeds)]
    divs = [100*(F - Is)/F for F in Fs]                      # divergence defined as (F_D - I_Schur)/F_D
    rows.append(dict(d=d_, I_Schur=Is, F_D=np.mean(Fs), divergence_pct=np.mean(divs),
                     div_se=np.std(divs, ddof=1)/np.sqrt(n_seeds) if n_seeds > 1 else np.nan))
tab2 = pd.DataFrame(rows).round(4)
print(tab2.to_string(index=False))
print("\nPaper (Table 2 / App. C):  d=0.05 -> 73.7% (I_Schur=0.047, F_D=0.180);  0.10 -> 42.0%;  0.30 -> 8.9%;  0.5-3.0 -> 0.6-3.1% (noise)")
tab2.to_csv(f"{OUT}/table2_schur_vs_exact.csv", index=False)

## 3. ZZHE convergence diagnostics (Appendix E) and synthetic cross-estimator validation (Appendix F, Table 4)
*Source:* `Untitled28.ipynb` cells 1–13, 15 — **VERBATIM**, except: (i) the Earthdata auth cell 0 is dropped (not used here);
(ii) **PATCHED** so the slow App-E sweeps (6, 6b–6e) only run when `QUICK=False`; (iii) `QUICK` shrinks the repeat counts in the Table-4 cells.
`RUN_APPE_SWEEPS` gates the sweeps. In `QUICK` mode the Table-4 numbers are only a smoke test.

In [ ]:
# ============================================================
# Cell 1 — Setup, wider prior support to eliminate boundary artifact
# ============================================================
import numpy as np
from scipy.optimize import minimize
from dataclasses import dataclass

_trapz = getattr(np, "trapezoid", None) or np.trapz

@dataclass
class ModelConfig:
    t_grid: np.ndarray
    sigma_pulse: float
    sigma_n: float
    w1: float
    w2: float
    c_range: tuple

CFG = ModelConfig(
    t_grid=np.linspace(-5, 5, 100),
    sigma_pulse=1.0,
    sigma_n=0.2,
    w1=0.4,
    w2=0.6,
    c_range=(-1.0, 1.0),
)

D_MAX_PRIOR = 4.0          # extended beyond the reported range (0.05-3.0)
D_GRID_REPORT = np.linspace(0.05, 3.0, 10)   # only interior points reported
H_GRID = np.linspace(0.01, D_MAX_PRIOR, 25)   # h can range up to the extended prior
C_GRID = np.linspace(CFG.c_range[0], CFG.c_range[1], 40)

In [ ]:
# ============================================================
# Cell 2 — Core model + marginal likelihood (unchanged from validated version)
# ============================================================
def pulse(t, tau, sigma_pulse):
    return np.exp(-(t - tau) ** 2 / (2 * sigma_pulse ** 2))

def mu_waveform(c, d, cfg):
    tau1, tau2 = c - d / 2.0, c + d / 2.0
    t = cfg.t_grid
    return cfg.w1 * pulse(t, tau1, cfg.sigma_pulse) + cfg.w2 * pulse(t, tau2, cfg.sigma_pulse)

def generate_noisy_waveform(c, d, cfg, rng):
    return mu_waveform(c, d, cfg) + rng.normal(0.0, cfg.sigma_n, size=cfg.t_grid.shape)

def neg_log_lik(params, x_waveform, cfg):
    c, d = params
    mu = mu_waveform(c, d, cfg)
    return 0.5 * np.sum((x_waveform - mu) ** 2) / cfg.sigma_n ** 2

def marginal_likelihood_mc(x_waveform, d, c_grid, cfg):
    t = cfg.t_grid[:, None]
    c = c_grid[None, :]
    tau1, tau2 = c - d / 2.0, c + d / 2.0
    mu = cfg.w1 * pulse(t, tau1, cfg.sigma_pulse) + cfg.w2 * pulse(t, tau2, cfg.sigma_pulse)
    resid = x_waveform[:, None] - mu
    log_lik = -0.5 * np.sum(resid ** 2, axis=0) / cfg.sigma_n ** 2
    max_ll = log_lik.max()
    L = np.exp(log_lik - max_ll).mean()
    return L, max_ll

In [ ]:
# ============================================================
# Cell 3 — ZZB floor via direct-MC P_err (validated version) + grid-convergence variant
# ============================================================
def p_err_direct_mc(d, h, cfg, c_grid, n_sim_per_hyp, rng):
    n_correct, n_total = 0, 2 * n_sim_per_hyp
    for hyp, d_true in [(0, d), (1, d + h)]:
        for _ in range(n_sim_per_hyp):
            c_true = rng.uniform(cfg.c_range[0], cfg.c_range[1])
            x = generate_noisy_waveform(c_true, d_true, cfg, rng)
            L0, max0 = marginal_likelihood_mc(x, d, c_grid, cfg)
            L1, max1 = marginal_likelihood_mc(x, d + h, c_grid, cfg)
            log_L0 = max0 + np.log(L0 + 1e-300)
            log_L1 = max1 + np.log(L1 + 1e-300)
            decision = 0 if log_L0 >= log_L1 else 1
            if decision == hyp:
                n_correct += 1
    return 1.0 - n_correct / n_total

def zzb_floor(d, cfg, h_grid, c_grid, n_sim_per_hyp, rng):
    p_errs = np.array([p_err_direct_mc(d, h, cfg, c_grid, n_sim_per_hyp, rng) for h in h_grid])
    integrand = h_grid * p_errs
    integrand_vf = np.maximum.accumulate(integrand[::-1])[::-1]
    return 0.5 * _trapz(integrand_vf, h_grid)

In [ ]:
# ============================================================
# Cell 4 — Three structurally different estimators
# ============================================================
def fit_local_mle(x, cfg, c_true, d_true):
    res = minimize(neg_log_lik, x0=[c_true, d_true], args=(x, cfg), method='Nelder-Mead')
    return res.x

def fit_global_mle(x, cfg, n_restarts, rng):
    best, best_cost = None, np.inf
    for _ in range(n_restarts):
        c0 = rng.uniform(*cfg.c_range)
        d0 = rng.uniform(0.05, 3.0)
        res = minimize(neg_log_lik, x0=[c0, d0], args=(x, cfg), method='Nelder-Mead')
        if res.fun < best_cost:
            best_cost, best = res.fun, res.x
    return best

def fit_grid_search(x, cfg, c_grid_search, d_grid_search):
    best_cost, best = np.inf, None
    for c0 in c_grid_search:
        for d0 in d_grid_search:
            mu = mu_waveform(c0, d0, cfg)
            cost = 0.5 * np.sum((x - mu) ** 2) / cfg.sigma_n ** 2
            if cost < best_cost:
                best_cost, best = cost, (c0, d0)
    # local polish
    res = minimize(neg_log_lik, x0=list(best), args=(x, cfg), method='Nelder-Mead')
    return res.x

C_GRID_SEARCH = np.linspace(CFG.c_range[0], CFG.c_range[1], 15)
D_GRID_SEARCH = np.linspace(0.05, 3.0, 20)

ESTIMATORS = {
    'local_mle': lambda x, c_true, d_true, rng: fit_local_mle(x, CFG, c_true, d_true),
    'global_mle': lambda x, c_true, d_true, rng: fit_global_mle(x, CFG, 6, rng),
    'grid_search': lambda x, c_true, d_true, rng: fit_grid_search(x, CFG, C_GRID_SEARCH, D_GRID_SEARCH),
}

In [ ]:
# ============================================================
# Cell 5 — Empirical MSE with CI, per estimator, per d, across multiple
# independent Monte Carlo repeat-batches (not just one big batch)
# ============================================================
def empirical_mse_with_ci(d, cfg, estimator_fn, n_repeats, n_trials_per_repeat, rng):
    repeat_means = []
    for _ in range(n_repeats):
        errs = []
        for _ in range(n_trials_per_repeat):
            c_true = rng.uniform(*cfg.c_range)
            x = generate_noisy_waveform(c_true, d, cfg, rng)
            c_hat, d_hat = estimator_fn(x, c_true, d, rng)
            errs.append((d_hat - d) ** 2)
        repeat_means.append(np.mean(errs))
    repeat_means = np.array(repeat_means)
    return repeat_means.mean(), repeat_means.std(ddof=1) / np.sqrt(n_repeats)  # mean, SE-of-mean

In [ ]:
# ============================================================
# Cell 6 — ZZB with its own repeat-based CI (grid-convergence embedded:
# run once at n_sim=100/25pts, once at n_sim=200/40pts, both must agree)
# ============================================================
def zzb_with_ci(d, cfg, h_grid, c_grid, n_sim_per_hyp, n_repeats, rng):
    reps = [zzb_floor(d, cfg, h_grid, c_grid, n_sim_per_hyp, rng) for _ in range(n_repeats)]
    reps = np.array(reps)
    return reps.mean(), reps.std(ddof=1) / np.sqrt(n_repeats)

if RUN_APPE_SWEEPS:
    rng = np.random.default_rng(1)

    print("Grid-convergence check at 3 representative d values ...")
    for d in [0.05, 1.0, 3.0]:
        mean_coarse, se_coarse = zzb_with_ci(d, CFG, np.linspace(0.01, D_MAX_PRIOR, 15), C_GRID,
                                              n_sim_per_hyp=60, n_repeats=4, rng=rng)
        mean_fine, se_fine = zzb_with_ci(d, CFG, np.linspace(0.01, D_MAX_PRIOR, 30), C_GRID,
                                          n_sim_per_hyp=120, n_repeats=4, rng=rng)
        diff = abs(mean_fine - mean_coarse)
        combined_se = np.sqrt(se_coarse**2 + se_fine**2)
        print(f"d={d}: coarse={mean_coarse:.5f}±{se_coarse:.5f}  fine={mean_fine:.5f}±{se_fine:.5f}  "
              f"within_2SE={diff <= 2*combined_se}")

In [ ]:
if RUN_APPE_SWEEPS:
    # ============================================================
    # Cell 6b — Diagnostic: does ZZB(d=0.05) converge monotonically as
    # h_grid density increases, holding n_sim and range fixed?
    # If yes -> confirms quadrature bias, not MC noise.
    # ============================================================
    rng = np.random.default_rng(2)

    h_densities = [15, 30, 60, 120, 240]
    print("h_grid density sweep at d=0.05 (n_sim_per_hyp fixed at 100, range fixed 0.01-4.0):")
    for n_pts in h_densities:
        h_grid_test = np.linspace(0.01, D_MAX_PRIOR, n_pts)
        mean_val, se_val = zzb_with_ci(0.05, CFG, h_grid_test, C_GRID,
                                         n_sim_per_hyp=100, n_repeats=4, rng=rng)
        print(f"  n_h_points={n_pts:4d}:  ZZB = {mean_val:.5f} ± {se_val:.5f}")

In [ ]:
def make_log_h_grid(h_min, h_max, n_pts):
    return np.geomspace(h_min, h_max, n_pts)

if RUN_APPE_SWEEPS:
    # ============================================================
    # Cell 6c — Fix: log-spaced h_grid concentrates resolution where the
    # P_err transition actually happens, instead of wasting points on
    # large-h where P_err is already ~0. Test with far fewer total points.
    # ============================================================
    rng = np.random.default_rng(3)


    log_densities = [20, 40, 80]
    print("LOG-SPACED h_grid sweep at d=0.05 (same n_sim_per_hyp=100, same range):")
    for n_pts in log_densities:
        h_grid_log = make_log_h_grid(0.005, D_MAX_PRIOR, n_pts)
        mean_val, se_val = zzb_with_ci(0.05, CFG, h_grid_log, C_GRID,
                                         n_sim_per_hyp=100, n_repeats=4, rng=rng)
        print(f"  n_h_points={n_pts:4d} (log-spaced):  ZZB = {mean_val:.5f} ± {se_val:.5f}")

    print("\nReference: uniform n=240 gave 0.12850 ± 0.00109 (best previous estimate)")

In [ ]:
if RUN_APPE_SWEEPS:
    # ============================================================
    # Cell 6d — Diagnostic: hold h_grid FIXED, increase n_sim_per_hyp only.
    # If the max-selection-bias hypothesis is correct, ZZB should DECREASE
    # and stabilize as per-point P_err noise shrinks, independent of grid density.
    # ============================================================
    rng = np.random.default_rng(4)

    h_grid_fixed = make_log_h_grid(0.005, D_MAX_PRIOR, 40)   # fixed grid, density held constant
    n_sim_sweep = [100, 200, 400, 800]

    print("n_sim_per_hyp sweep at d=0.05 (h_grid FIXED at 40 log-spaced points):")
    for n_sim in n_sim_sweep:
        mean_val, se_val = zzb_with_ci(0.05, CFG, h_grid_fixed, C_GRID,
                                         n_sim_per_hyp=n_sim, n_repeats=4, rng=rng)
        print(f"  n_sim_per_hyp={n_sim:4d}:  ZZB = {mean_val:.5f} ± {se_val:.5f}")

In [ ]:
if RUN_APPE_SWEEPS:
    # ============================================================
    # Cell 6e — One more point: confirm the decrease is converging,
    # not open-ended. If it plateaus, we lock in final settings.
    # ============================================================
    rng = np.random.default_rng(5)

    mean_1600, se_1600 = zzb_with_ci(0.05, CFG, h_grid_fixed, C_GRID,
                                       n_sim_per_hyp=1600, n_repeats=4, rng=rng)
    print(f"n_sim_per_hyp=1600:  ZZB = {mean_1600:.5f} ± {se_1600:.5f}")
    print(f"\nFull trend: 100->0.11604, 200->0.11525, 400->0.11288, 800->0.11009, 1600->{mean_1600:.5f}")
    print(f"Deltas: {0.11604-0.11525:.5f}, {0.11525-0.11288:.5f}, {0.11288-0.11009:.5f}, {0.11009-mean_1600:.5f}")


In [ ]:
# ============================================================
# Cell 7 — FULL locked sweep: 10 d-points x 3 estimators, with CI,
# using the converged settings (log h_grid=40, n_sim_per_hyp=800)
# ============================================================
rng = np.random.default_rng(10)

H_GRID_FINAL = make_log_h_grid(0.005, D_MAX_PRIOR, 40)
N_SIM_FINAL = 800 if not QUICK else 60
N_REPEATS_ZZB = 5 if not QUICK else 2          # for ZZB's own CI
N_REPEATS_EMP = 5 if not QUICK else 2          # for each estimator's empirical CI
N_TRIALS_PER_REPEAT = 20 if not QUICK else 5   # keep runtime bounded; this is the heaviest cell so far

print("Computing ZZB floor (with CI) at all 10 d-points ...")
zzb_means, zzb_ses = [], []
for d in D_GRID_REPORT:
    m, se = zzb_with_ci(d, CFG, H_GRID_FINAL, C_GRID, N_SIM_FINAL, N_REPEATS_ZZB, rng)
    zzb_means.append(m); zzb_ses.append(se)
    print(f"  d={d:.3f}  ZZB={m:.5f} ± {se:.5f}")

zzb_means, zzb_ses = np.array(zzb_means), np.array(zzb_ses)


In [ ]:
# ============================================================
# Cell 8 — Empirical MSE + CI for all 3 estimators at all 10 d-points
# ============================================================
print("\nComputing empirical MSE (with CI) for 3 estimators ...")
results = {name: {'mean': [], 'se': []} for name in ESTIMATORS}

for d in D_GRID_REPORT:
    for name, fn in ESTIMATORS.items():
        m, se = empirical_mse_with_ci(d, CFG, fn, N_REPEATS_EMP, N_TRIALS_PER_REPEAT, rng)
        results[name]['mean'].append(m)
        results[name]['se'].append(se)
    print(f"  d={d:.3f} done: " + ", ".join(
        f"{name}={results[name]['mean'][-1]:.5f}±{results[name]['se'][-1]:.5f}" for name in ESTIMATORS))

for name in ESTIMATORS:
    results[name]['mean'] = np.array(results[name]['mean'])
    results[name]['se'] = np.array(results[name]['se'])

In [ ]:
# ============================================================
# Cell 9 — Statistical-margin falsifiability check (2-SE separation,
# not just point-comparison) across all estimators
# ============================================================
print("\nFalsifiability check: ZZB + 2*SE_zzb <= current - 2*SE_current ?")
print(f"{'d':>6} {'estimator':>12} {'ZZB(upper)':>11} {'current(lower)':>15} {'margin_pass':>12}")

all_pass = True
for i, d in enumerate(D_GRID_REPORT):
    zzb_upper = zzb_means[i] + 2 * zzb_ses[i]
    for name in ESTIMATORS:
        cur_lower = results[name]['mean'][i] - 2 * results[name]['se'][i]
        passed = zzb_upper <= cur_lower
        all_pass &= passed
        print(f"{d:6.3f} {name:>12} {zzb_upper:11.5f} {cur_lower:15.5f} {str(passed):>12}")

print(f"\nALL points, ALL estimators pass with 2-SE margin: {all_pass}")

In [ ]:
# ============================================================
# Cell 10 — Final plot: single ZZB floor, three estimators overlaid
# (this is the headline figure for the paper's application section)
# ============================================================
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({'font.family': 'serif', 'font.size': 10, 'savefig.dpi': 300, 'savefig.bbox': 'tight'})

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(D_GRID_REPORT, zzb_means, '-', color='#2ca02c', linewidth=1.8, label='ZZB floor (estimator-agnostic)')
ax.fill_between(D_GRID_REPORT, zzb_means - 2*zzb_ses, zzb_means + 2*zzb_ses, color='#2ca02c', alpha=0.2)

colors = {'local_mle': '#1f77b4', 'global_mle': '#ff7f0e', 'grid_search': '#9467bd'}
markers = {'local_mle': 's', 'global_mle': 'o', 'grid_search': '^'}
for name in ESTIMATORS:
    ax.errorbar(D_GRID_REPORT, results[name]['mean'], yerr=2*results[name]['se'],
                fmt=markers[name], color=colors[name], markersize=5, capsize=2, label=name)

ax.set_yscale('log')
ax.set_xlabel('separation $d$')
ax.set_ylabel('MSE (log scale, $\\pm2$SE)')
ax.legend(fontsize=8, frameon=False)
plt.tight_layout()
plt.savefig('zzb_pillar_result.png', dpi=300)
plt.show()

In [ ]:
# ============================================================
# Cell 11 — Targeted fixes for the 6 failing combos, NOT a full rerun
# ============================================================
rng = np.random.default_rng(20)

# --- Fix 1: de-alias grid_search's candidate grid from the test-point grid ---
D_GRID_SEARCH_FIXED = np.linspace(0.05, 3.0, 20) + 0.5 * (3.0 - 0.05) / 20 / 2  # half-cell offset
D_GRID_SEARCH_FIXED = D_GRID_SEARCH_FIXED[D_GRID_SEARCH_FIXED <= 3.0]

def fit_grid_search_fixed(x, cfg, c_grid_search, d_grid_search):
    best_cost, best = np.inf, None
    for c0 in c_grid_search:
        for d0 in d_grid_search:
            mu = mu_waveform(c0, d0, cfg)
            cost = 0.5 * np.sum((x - mu) ** 2) / cfg.sigma_n ** 2
            if cost < best_cost:
                best_cost, best = cost, (c0, d0)
    res = minimize(neg_log_lik, x0=list(best), args=(x, cfg), method='Nelder-Mead')
    return res.x

grid_search_fixed_fn = lambda x, c_true, d_true, rng: fit_grid_search_fixed(x, CFG, C_GRID_SEARCH, D_GRID_SEARCH_FIXED)

print("Re-checking d=0.050, grid_search (de-aliased grid) ...")
m, se = empirical_mse_with_ci(0.050, CFG, grid_search_fixed_fn, N_REPEATS_EMP, N_TRIALS_PER_REPEAT, rng)
cur_lower = m - 2 * se
print(f"  new: mean={m:.5f} se={se:.5f}  lower={cur_lower:.5f}  zzb_upper={zzb_means[0]+2*zzb_ses[0]:.5f}  pass={zzb_means[0]+2*zzb_ses[0] <= cur_lower}")

# --- Fix 2: higher precision at the 3 borderline-tight points ---
N_REPEATS_HIGH = 15 if not QUICK else 3
N_TRIALS_HIGH = 40 if not QUICK else 5

borderline = [(1.033, 'local_mle', 3), (1.689, 'local_mle', 5), (2.344, 'global_mle', 7), (2.344, 'grid_search', 7)]
print("\nRe-checking borderline points with higher precision ...")
for d, name, idx in borderline:
    m, se = empirical_mse_with_ci(d, CFG, ESTIMATORS[name], N_REPEATS_HIGH, N_TRIALS_HIGH, rng)
    cur_lower = m - 2 * se
    zzb_upper = zzb_means[idx] + 2 * zzb_ses[idx]
    print(f"  d={d} {name}: mean={m:.5f} se={se:.5f} lower={cur_lower:.5f} zzb_upper={zzb_upper:.5f} pass={zzb_upper <= cur_lower}")

# --- Fix 3: d=1.689 global_mle -- heavy-tail characterization, not a floor test ---
print("\nCharacterizing d=1.689 global_mle's heavy tail (large sample, single batch) ...")
errs_1689 = []
for _ in range(300 if not QUICK else 40):
    c_true = rng.uniform(*CFG.c_range)
    x = generate_noisy_waveform(c_true, 1.689, CFG, rng)
    c_hat, d_hat = fit_global_mle(x, CFG, 6, rng)
    errs_1689.append((d_hat - 1.689) ** 2)
errs_1689 = np.array(errs_1689)
capture_thresh = np.median(errs_1689) * 10   # crude outlier flag
n_captures = (errs_1689 > capture_thresh).sum()
print(f"  n=300: median_err={np.median(errs_1689):.5f}  mean_err={errs_1689.mean():.5f}  "
      f"flagged-outliers={n_captures} ({100*n_captures/300:.1f}%)  "
      f"mean_excl_outliers={errs_1689[errs_1689<=capture_thresh].mean():.5f}")

In [ ]:
# ---------- Tally for Table 4: what the *primary locked run* gives vs. what the paper reports (29/30) ----------
prim = []
for i, d_ in enumerate(D_GRID_REPORT):
    zu = zzb_means[i] + 2*zzb_ses[i]
    for name in ESTIMATORS:
        lo = results[name]["mean"][i] - 2*results[name]["se"][i]
        prim.append(dict(d=round(float(d_), 3), estimator=name, mse=results[name]["mean"][i], se=results[name]["se"][i], zzb_upper=zu, margin_lower=lo, pass_=bool(zu <= lo)))
prim = pd.DataFrame(prim)
print("Primary run margin passes: %d / %d" % (prim.pass_.sum(), len(prim)))
print(prim[~prim.pass_].round(5).to_string(index=False))
print("\nPaper reports 29/30. Difference = rows above that were re-run at higher precision ONLY because they failed "
      "(1.033 local, 1.689 local, 2.344 global, 2.344 grid), plus 1.689 global_mle which was handled as a 'heavy-tail characterisation'.")
print("Table 4 in the paper prints the ORIGINAL mean +- SE for those rows, so the printed table alone does not show 29/30.")
Table4 = prim.pivot(index="d", columns="estimator", values="mse").round(3); print(Table4)
prim.to_csv(f"{OUT}/table4_primary_run.csv", index=False)

## 4. Noise-level robustness of the ZZHE floor  (Appendix F, Table 5, Fig. 5)
*Source:* `Untitled7 (2).ipynb` cells 0, 1, 3 — **PATCHED** (output path `/content/…` → `./…`; `QUICK` shrinks n_c, n_sim, trials).
Logged result (locked precision n_c=800, n_sim=800, 6 restarts, ~31 min): 24/24 pass. Cell 1 is the targeted re-run of the borderline point (σ_n=0.5, d=2.5).

In [ ]:
import numpy as np
from scipy.optimize import minimize
from scipy.special import logsumexp
import json, time

rng = np.random.default_rng(0)

# ---- Model (matches paper's ModelConfig: sigma_pulse=1.0, w1=0.4, w2=0.6) ----
sigma_pulse = 1.0
w1, w2 = 0.4, 0.6
T_GRID = np.linspace(-8, 8, 320)
C_RANGE = (-2.0, 2.0)
N_C = 800 if not QUICK else 100

def pulse(t, tau, sigma):
    return np.exp(-(t - tau) ** 2 / (2 * sigma ** 2))

def mean_waveform(c, d):
    c = np.atleast_1d(c)[..., None]
    d = np.atleast_1d(d)[..., None]
    tau1 = c - d / 2
    tau2 = c + d / 2
    return w1 * pulse(T_GRID, tau1, sigma_pulse) + w2 * pulse(T_GRID, tau2, sigma_pulse)

C_GRID = np.linspace(*C_RANGE, N_C)

def template_bank(d):
    return mean_waveform(C_GRID, d)

def marginal_loglik(X, templates, sigma_n, chunk=100):
    # chunked over samples so N_C=800 doesn't blow up memory at n_sim=800
    n = X.shape[0]
    out = np.empty(n)
    for start in range(0, n, chunk):
        end = min(start + chunk, n)
        diff = X[start:end, None, :] - templates[None, :, :]
        sq = np.sum(diff ** 2, axis=-1) / (2 * sigma_n ** 2)
        out[start:end] = logsumexp(-sq, axis=1) - np.log(templates.shape[0])
    return out

def p_err_direct_mc(d, h, sigma_n, n_sim, tmpl_d, tmpl_dh):
    c0 = rng.uniform(*C_RANGE, size=n_sim)
    x0 = mean_waveform(c0, d)[:, 0, :] if mean_waveform(c0, d).ndim == 3 else mean_waveform(c0, d)
    x0 = x0 + rng.normal(0, sigma_n, size=x0.shape)
    ll_d0 = marginal_loglik(x0, tmpl_d, sigma_n)
    ll_dh0 = marginal_loglik(x0, tmpl_dh, sigma_n)
    err0 = np.mean(ll_dh0 > ll_d0)

    c1 = rng.uniform(*C_RANGE, size=n_sim)
    x1 = mean_waveform(c1, d + h)
    if x1.ndim == 3:
        x1 = x1[:, 0, :]
    x1 = x1 + rng.normal(0, sigma_n, size=x1.shape)
    ll_d1 = marginal_loglik(x1, tmpl_d, sigma_n)
    ll_dh1 = marginal_loglik(x1, tmpl_dh, sigma_n)
    err1 = np.mean(ll_d1 > ll_dh1)

    return 0.5 * (err0 + err1)

def l_zzb(d, sigma_n, h_grid, n_sim, tmpl_cache):
    tmpl_d = tmpl_cache[round(d, 6)]
    g = np.zeros_like(h_grid)
    for i, h in enumerate(h_grid):
        dh = d + h
        key = round(dh, 6)
        if key not in tmpl_cache:
            tmpl_cache[key] = template_bank(dh)
        tmpl_dh = tmpl_cache[key]
        pe = p_err_direct_mc(d, h, sigma_n, n_sim, tmpl_d, tmpl_dh)
        g[i] = h * pe
    g_hat = np.maximum.accumulate(g[::-1])[::-1]
    return 0.5 * np.trapezoid(g_hat, h_grid)

def single_waveform(c, d):
    return mean_waveform(np.array([c]), np.array([d]))[0, :]

def neg_log_lik(params, x, sigma_n):
    c, d = params
    mean = single_waveform(c, d)
    return np.sum((x - mean) ** 2) / (2 * sigma_n ** 2)

def empirical_mse(d, sigma_n, n_trials, restarts=6):
    errs = []
    for _ in range(n_trials):
        c_true = rng.uniform(*C_RANGE)
        mean = single_waveform(c_true, d)
        x = mean + rng.normal(0, sigma_n, size=mean.shape)
        best = None
        for _ in range(restarts):
            c0 = rng.uniform(*C_RANGE)
            d0 = np.clip(d + rng.normal(0, 0.5), 0.01, 3.5)
            res = minimize(neg_log_lik, x0=[c0, d0], args=(x, sigma_n), method='Nelder-Mead')
            if best is None or res.fun < best.fun:
                best = res
        d_hat = best.x[1]
        errs.append((d_hat - d) ** 2)
    return float(np.mean(errs)), float(np.std(errs) / np.sqrt(n_trials))

def snr_db(sigma_n, d=1.0, c=0.0):
    mean = single_waveform(c, d)
    sig_power = np.mean(mean ** 2)
    return 10 * np.log10(sig_power / sigma_n ** 2)


if __name__ == "__main__":
    t0 = time.time()
    d_grid = [0.10, 0.30, 0.60, 1.00, 1.70, 2.50]
    sigma_n_grid = [0.10, 0.20, 0.35, 0.50]
    h_grid = np.geomspace(0.02, 4.0, 12)
    n_sim_zzb = 800 if not QUICK else 100
    n_trials_mle = 60 if not QUICK else 10

    results = []
    for sigma_n in sigma_n_grid:
        tmpl_cache = {}
        for d in d_grid:
            if round(d, 6) not in tmpl_cache:
                tmpl_cache[round(d, 6)] = template_bank(d)
            zzb = l_zzb(d, sigma_n, h_grid, n_sim_zzb, tmpl_cache)
            mse, mse_se = empirical_mse(d, sigma_n, n_trials_mle)
            row = dict(sigma_n=sigma_n, snr_db=snr_db(sigma_n, d=d), d=d,
                       L_ZZB=zzb, MSE=mse, MSE_SE=mse_se, pass_margin=bool(zzb <= mse))
            results.append(row)
            print(f"sigma_n={sigma_n:.2f} SNR={row['snr_db']:.1f}dB d={d:.2f}  "
                  f"L_ZZB={zzb:.5f}  MSE={mse:.5f}+-{mse_se:.5f}  pass={row['pass_margin']}")

    with open('zzhe_noise_sweep_results.json', 'w') as f:
        json.dump(results, f, indent=2)
    print(f"\nTotal time: {time.time()-t0:.1f}s")

In [ ]:
# Targeted re-run of the one borderline point, higher precision only here
rng = np.random.default_rng(1)   # fresh seed
d, sigma_n = 2.50, 0.50
tmpl_cache = {round(d, 6): template_bank(d)}

zzb = l_zzb(d, sigma_n, h_grid, n_sim_zzb, tmpl_cache)          # keep n_sim_zzb=800 as-is
mse, mse_se = empirical_mse(d, sigma_n, n_trials=(200 if not QUICK else 20), restarts=6)  # n_trials: 60 -> 200

print(f"d={d} sigma_n={sigma_n}")
print(f"L_ZZB={zzb:.5f}")
print(f"MSE={mse:.5f} +- {mse_se:.5f}")
print(f"MSE - 2*SE = {mse - 2*mse_se:.5f}")
print(f"margin check: {'PASS' if zzb <= mse - 2*mse_se else 'FAIL'}")

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

with open('zzhe_noise_sweep_results.json') as f:
    results = json.load(f)

sigma_n_grid = sorted(set(r['sigma_n'] for r in results))
d_grid = sorted(set(r['d'] for r in results))

# viridis-family colors, consistent with the rest of your paper's palette
cmap = plt.cm.viridis
sigma_colors = {s: cmap(i/(len(sigma_n_grid)-1)) for i, s in enumerate(sigma_n_grid)}
d_colors = {d: cmap(i/(len(d_grid)-1)) for i, d in enumerate(d_grid)}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ---- Panel (a): MSE (solid) and L_ZZB (dashed) vs separation, one line per noise level ----
ax = axes[0]
for s in sigma_n_grid:
    rows = sorted([r for r in results if r['sigma_n'] == s], key=lambda r: r['d'])
    ds = [r['d'] for r in rows]
    mse = [r['MSE'] for r in rows]
    mse_se = [r['MSE_SE'] for r in rows]
    zzb = [r['L_ZZB'] for r in rows]
    snr_label = np.mean([r['snr_db'] for r in rows])
    ax.errorbar(ds, mse, yerr=[2*se for se in mse_se], color=sigma_colors[s], marker='o',
                lw=1.8, capsize=3, label=f"MSE, SNR≈{snr_label:.0f}dB")
    ax.plot(ds, zzb, color=sigma_colors[s], marker='x', ls='--', lw=1.8)
ax.set_yscale('log')
ax.set_xlabel('separation $d$')
ax.set_ylabel('MSE / $L_{ZZB}$')
ax.set_title('(a) MSE (solid) and $L_{ZZB}$ (dashed) vs. separation\n[locked precision: $n_c$=800, $n_{sim}$=800, 6 restarts]', fontsize=10)
ax.legend(fontsize=7)

# ---- Panel (b): same data, vs SNR, one line per separation ----
ax = axes[1]
for d in d_grid:
    rows = sorted([r for r in results if r['d'] == d], key=lambda r: r['snr_db'])
    snr = [r['snr_db'] for r in rows]
    mse = [r['MSE'] for r in rows]
    mse_se = [r['MSE_SE'] for r in rows]
    zzb = [r['L_ZZB'] for r in rows]
    ax.errorbar(snr, mse, yerr=[2*se for se in mse_se], color=d_colors[d], marker='o',
                lw=1.8, capsize=3, label=f"MSE, d={d:.2f}")
    ax.plot(snr, zzb, color=d_colors[d], marker='x', ls='--', lw=1.8)
ax.set_yscale('log')
ax.set_xlabel('SNR (dB)')
ax.set_ylabel('MSE / $L_{ZZB}$')
ax.set_title('(b) MSE (solid) and $L_{ZZB}$ (dashed) vs. SNR\n[locked precision: $n_c$=800, $n_{sim}$=800, 6 restarts]', fontsize=10)
ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig('noise_sweep_locked_precision.pdf', dpi=300)
plt.show()

## 5. Cross-architecture convergence to the ZZHE floor  (Appendix D, Table 3, Fig. 4)
*FIXED.* `Untitled7 (2).ipynb` cell 2 never finished (KeyboardInterrupt during CNN training) and the plotting cells crash (`results` is the noise-sweep list, not the
per-estimator dict). **No uploaded notebook contains the numbers printed in the paper's Table 3**, so that table cannot be traced to a logged run;
re-running this cell produces replacement numbers (they will differ from the paper's — update the table from this output).

In [ ]:
# ================= Experiment: cross-architecture convergence to the ZZHE floor (Appendix D, Table 3, Fig. 4) =================
# The version inside Untitled7 never finished (KeyboardInterrupt in CNN training; later cells crash on `results` being a list).
# Fixed here: results stored in conv_results; vectorised data generation; floor read from the noise-sweep json if available.
import torch, torch.nn as nn, torch.optim as optim
from scipy.optimize import minimize
dev = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0); rngA = np.random.default_rng(0)

sig_p, W1A, W2A = 1.0, 0.4, 0.6
TG = np.linspace(-8, 8, 320); CR = (-2.0, 2.0); SNA = 0.20
N_TRAIN = 4000 if QUICK else 20000
EPOCHS  = 30   if QUICK else 300
N_TEST_PER_D = 10 if QUICK else 40
D_TEST = [0.10, 0.30, 0.60, 1.00, 1.70, 2.50]

def sample_batch(n, d_range=(0.05, 3.0)):
    d = rngA.uniform(*d_range, n); c = rngA.uniform(*CR, n)
    X = mu_cd(c, d, TG, sig_p, W1A, W2A) + rngA.normal(0, SNA, (n, len(TG)))
    return X.astype(np.float32), d.astype(np.float32)

X_tr, d_tr = sample_batch(N_TRAIN)
X_tr_t = torch.tensor(X_tr, device=dev); d_tr_t = torch.tensor(d_tr, device=dev).unsqueeze(1)

def train_torch_model(model, X, y, epochs=EPOCHS, lr=1e-3, bs=256):
    model.to(dev); opt = optim.Adam(model.parameters(), lr=lr); lf = nn.MSELoss(); n = X.shape[0]
    for ep in range(epochs):
        perm = torch.randperm(n, device=dev)
        for i in range(0, n, bs):
            idx = perm[i:i+bs]; opt.zero_grad(); loss = lf(model(X[idx]), y[idx]); loss.backward(); opt.step()
    return model.eval()

class MLP(nn.Module):
    def __init__(s, n):
        super().__init__(); s.net = nn.Sequential(nn.Linear(n,128), nn.ReLU(), nn.Linear(128,64), nn.ReLU(), nn.Linear(64,1))
    def forward(s, x): return s.net(x)
class CNN1D(nn.Module):
    def __init__(s, n):
        super().__init__()
        s.conv = nn.Sequential(nn.Conv1d(1,16,9,padding=4), nn.ReLU(), nn.MaxPool1d(2), nn.Conv1d(16,32,9,padding=4), nn.ReLU(), nn.MaxPool1d(2))
        s.fc = nn.Sequential(nn.Linear(32*(n//4),64), nn.ReLU(), nn.Linear(64,1))
    def forward(s, x): return s.fc(s.conv(x.unsqueeze(1)).flatten(1))
class FourierMLP(nn.Module):                                  # "neural-field family" (Tancik et al. 2020)
    def __init__(s, n, nf=64, sf=5.0):
        super().__init__(); s.register_buffer("B", torch.randn(n, nf)*sf)
        s.net = nn.Sequential(nn.Linear(2*nf,128), nn.ReLU(), nn.Linear(128,64), nn.ReLU(), nn.Linear(64,1))
    def forward(s, x):
        p = x @ s.B; return s.net(torch.cat([torch.sin(p), torch.cos(p)], -1))

models = {}
for name, cls in [("MLP", MLP), ("CNN", CNN1D), ("FourierMLP", FourierMLP)]:
    t0 = time.time(); models[name] = train_torch_model(cls(len(TG)), X_tr_t, d_tr_t); print(f"{name}: trained in {time.time()-t0:.0f}s")

def nll(p, x): return np.sum((x - mu_cd(p[0], p[1], TG, sig_p, W1A, W2A)[0])**2) / (2*SNA**2)
def mle_est(x, restarts=6):
    best = None
    for _ in range(restarts):
        r = minimize(nll, x0=[rngA.uniform(*CR), rngA.uniform(0.05, 3.0)], args=(x,), method="Nelder-Mead")
        if best is None or r.fun < best.fun: best = r
    return best.x[1]

DG = np.linspace(0.05, 3.0, 60); CG = np.linspace(*CR, 60)
TEMPL = np.stack([mu_cd(CG, np.full(len(CG), dd), TG, sig_p, W1A, W2A) for dd in DG])         # (nd, nc, nt)
def oracle_bayes(x):                                            # posterior mean of d under the known model, c marginalised
    lp = -np.sum((TEMPL - x[None, None, :])**2, -1) / (2*SNA**2); lp -= lp.max()
    p = np.exp(lp); p /= p.sum(); return float((p.sum(1)*DG).sum())

conv_results = {k: [] for k in ["MLP", "CNN", "FourierMLP", "MLE", "Oracle"]}
for d_ in D_TEST:
    errs = {k: [] for k in conv_results}
    for _ in range(N_TEST_PER_D):
        c_true = rngA.uniform(*CR)
        x = mu_cd(c_true, d_, TG, sig_p, W1A, W2A)[0] + rngA.normal(0, SNA, len(TG))
        xt = torch.tensor(x.astype(np.float32), device=dev).unsqueeze(0)
        with torch.no_grad():
            for k in ["MLP", "CNN", "FourierMLP"]: errs[k].append((models[k](xt).item() - d_)**2)
        errs["MLE"].append((mle_est(x) - d_)**2); errs["Oracle"].append((oracle_bayes(x) - d_)**2)
    for k in conv_results: conv_results[k].append((d_, np.mean(errs[k]), np.std(errs[k], ddof=1)/np.sqrt(N_TEST_PER_D)))
    print(f"d={d_:.2f}: " + ", ".join(f"{k}={np.mean(errs[k]):.4f}" for k in conv_results))

# floor at sigma_n=0.20 : from the noise-sweep json if Section 4 was run, else the values printed in the paper's Table 5
try:
    _r = [r for r in json.load(open("zzhe_noise_sweep_results.json")) if abs(r["sigma_n"] - 0.20) < 1e-9]
    L_ZZB_by_d = {round(r["d"], 2): r["L_ZZB"] for r in _r}
except Exception:
    L_ZZB_by_d = {0.10: 0.06573, 0.30: 0.04200, 0.60: 0.02409, 1.00: 0.01487, 1.70: 0.00966, 2.50: 0.00879}
    print("(using Table-5 floor values from the paper)")

tab3 = pd.DataFrame({k: [v[1] for v in conv_results[k]] for k in conv_results}, index=D_TEST); tab3.index.name = "d"
tab3["L_ZZB"] = [L_ZZB_by_d[round(d_, 2)] for d_ in D_TEST]
print("\nTable 3 (regenerated):"); print(tab3.round(4).to_string())
viol = [(k, d_) for k in conv_results for (d_, m, se) in conv_results[k] if m + 2*se < L_ZZB_by_d[round(d_, 2)]]
print("\nCombinations with MSE+2SE below the floor (would contradict Theorem/ZZHE):", viol or "none")
tab3.to_csv(f"{OUT}/table3_architectures.csv")

fig, ax = plt.subplots(figsize=(6.5, 4.5))
mk = dict(MLP="o", CNN="s", FourierMLP="^", MLE="D", Oracle="*")
for k, v in conv_results.items():
    ax.errorbar([a[0] for a in v], [a[1] for a in v], yerr=[2*a[2] for a in v], marker=mk[k], capsize=3, label=k)
ax.plot(list(L_ZZB_by_d), list(L_ZZB_by_d.values()), "k--x", lw=2.5, label="L_ZZB(d) floor")
ax.axvline(KAPPA*sig_p, color="gray", ls=":"); ax.set_yscale("log"); ax.set_xlabel("separation d"); ax.set_ylabel("MSE"); ax.legend(fontsize=8)
ax.set_title("All representations converge toward the same information ceiling")
plt.tight_layout(); plt.savefig(f"{OUT}/fig4_convergence.png", dpi=200); plt.show()

## 6. Encoder-probe grounding  (Appendix D, Fig. 3)
*GENERATED.* Not in any uploaded notebook. Paper's numbers: separation-probe MAE 0.585 → 0.335 (−43%), global-denominator R² 0.72 → 0.91, centroid-control R² > 0.98.
Whatever this produces is what you should report — if the control probe is not flat, or the separation probe does not improve, the appendix claim needs rewording, not tuning.

In [ ]:
# ================= Experiment: encoder-probe grounding (Appendix D, Fig. 3) =================
# NOT present in any uploaded notebook -> generated from the paper's description:
#  10,000 synthetic two-sub-point waveforms; shared MLP encoder trained by unsupervised reconstruction; frozen;
#  linear probe for d = tau2 - tau1, second linear probe for the centroid c as a control; latent dims 4..32.
# ASSUMPTIONS (not stated in the paper): sigma_n=0.2, centroid ~U[-2,2], d/tau_thr ~U[0.05,3], 70/30 split, ridge probe,
# "global-denominator R^2" := 1 - MSE_bin / Var_total(d).  Uses sklearn MLP (no torch needed).
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import Ridge

SPb, SNb, W1b, W2b = 1.0, 0.2, 0.4, 0.6
TGb = np.linspace(-8, 8, 128)
TAU_THR = KAPPA*SPb
N_WAVE = 3000 if QUICK else 10000
rngP = np.random.default_rng(11)
ratio = rngP.uniform(0.05, 3.0, N_WAVE); dP = ratio*TAU_THR; cP = rngP.uniform(-2, 2, N_WAVE)
XP = mu_cd(cP, dP, TGb, SPb, W1b, W2b) + rngP.normal(0, SNb, (N_WAVE, len(TGb)))
ntr = int(0.7*N_WAVE)

def encode(model, X, n_enc_layers):
    """frozen encoder = first n_enc_layers hidden layers of the trained autoencoder (ReLU units, as trained)."""
    h = X
    for W, b in list(zip(model.coefs_, model.intercepts_))[:n_enc_layers]:
        h = np.maximum(h @ W + b, 0)
    return h

bins = np.arange(0.0, 3.01, 0.5); mids = 0.5*(bins[:-1] + bins[1:])
res_probe = {}
for k in ([4, 16] if QUICK else [4, 8, 16, 32]):
    ae = MLPRegressor(hidden_layer_sizes=(64, k, 64), activation="relu", max_iter=60 if QUICK else 300, random_state=0, early_stopping=False)
    ae.fit(XP[:ntr], XP[:ntr])                                          # unsupervised reconstruction
    Z = encode(ae, XP, 2)                                               # frozen encoder: 128 -> 64 -> k
    pd_ = Ridge(alpha=1e-3).fit(Z[:ntr], dP[:ntr]); pc_ = Ridge(alpha=1e-3).fit(Z[:ntr], cP[:ntr])
    te = np.arange(ntr, N_WAVE); ed = np.abs(pd_.predict(Z[te]) - dP[te]); ec = np.abs(pc_.predict(Z[te]) - cP[te])
    varD = dP[te].var()
    rows = []
    for lo, hi, m in zip(bins[:-1], bins[1:], mids):
        sel = (ratio[te] >= lo) & (ratio[te] < hi)
        rows.append(dict(ratio_mid=m, n=int(sel.sum()), MAE_d=ed[sel].mean(), MAE_c=ec[sel].mean(),
                         R2_d_global=1 - (ed[sel]**2).mean()/varD))
    res_probe[k] = pd.DataFrame(rows)
    r = res_probe[k]
    print(f"latent={k:>2}: MAE_d {r.MAE_d.iloc[0]:.3f} -> {r.MAE_d.iloc[-1]:.3f} (reduction {100*(1-r.MAE_d.iloc[-1]/r.MAE_d.iloc[0]):.0f}%), "
          f"R2_global {r.R2_d_global.iloc[0]:.2f} -> {r.R2_d_global.iloc[-1]:.2f}, centroid MAE range {r.MAE_c.min():.3f}-{r.MAE_c.max():.3f}")
print("\nPaper: separation-probe MAE 0.585 -> 0.335 (-43%), R2 0.72 -> 0.91, centroid control flat (R2>0.98).")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.4)); kk = list(res_probe)[-1]; r = res_probe[kk]
ax[0].plot(r.ratio_mid, r.MAE_d, "o-", label="MAE (separation)"); ax[0].plot(r.ratio_mid, r.MAE_c, "s-", label="MAE (centroid, control)")
ax[1].plot(r.ratio_mid, r.R2_d_global, "o-", label="R2 (separation, global denominator)")
for a in ax: a.axvline(1.0, color="r", ls=":"); a.set_xlabel("separation / threshold"); a.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/fig3_encoder_probe.png", dpi=200); plt.show()

## 7. Loss-landscape observation: swap basin for M = 2  (Appendix A)
*GENERATED.* Not in the zip (`Untitled23/25` are CRLB-blow-up calibration and a swap-floor sketch, not this experiment).
The 1–8% capture range and the 0.6–0.8 onset in the paper cannot be traced to a logged run.

In [ ]:
# ================= Experiment: M=2 loss-landscape / swap basin (Appendix A) =================
# NOT present in any uploaded notebook -> generated from the paper's description (multi-start Adam, 200 starts, w=(0.4,0.6)).
# Implemented in numpy with analytic gradients (independent starts, batched). Two initialisations are reported because the
# paper does not say which was used: 'uniform' (global random) and 'near' (around the true centroid).
def adam_multistart(d, sigma=1.0, w=(0.4, 0.6), sn=0.05, n_starts=200, steps=1500, lr=0.05, init="uniform", seed=0):
    rr = np.random.default_rng(seed); t = np.linspace(-8, 8, 320)
    tt = np.array([-d/2, d/2]); x = w[0]*P_(t, tt[0], sigma) + w[1]*P_(t, tt[1], sigma) + rr.normal(0, sn, len(t))
    if init == "uniform":     th = rr.uniform(-4, 4, (n_starts, 2))
    elif init == "near":      th = rr.normal(0, 1.0*sigma, (n_starts, 2))            # around the true centroid
    else:                     th = tt[None, :] + rr.normal(0, 0.5*sigma, (n_starts, 2))   # 'perturbed': truth + noise
    m = np.zeros_like(th); v = np.zeros_like(th); b1, b2, eps = 0.9, 0.999, 1e-8
    for i in range(1, steps+1):
        p1 = P_(t[None], th[:, :1], sigma); p2 = P_(t[None], th[:, 1:], sigma)
        r = w[0]*p1 + w[1]*p2 - x[None]
        # dP_ is d/dtau of P; loss = 0.5*sum(r^2)/sn^2 -> grad_tau = sum(r * w * dP/dtau)/sn^2
        g = np.stack([(r*w[0]*dP_(t[None], th[:, :1], sigma)).sum(1), (r*w[1]*dP_(t[None], th[:, 1:], sigma)).sum(1)], 1) / sn**2
        m = b1*m + (1-b1)*g; v = b2*v + (1-b2)*g*g
        th = th - lr*(m/(1-b1**i))/(np.sqrt(v/(1-b2**i)) + eps)
    r = w[0]*P_(t[None], th[:, :1], sigma) + w[1]*P_(t[None], th[:, 1:], sigma) - x[None]
    cost = 0.5*(r**2).sum(1)/sn**2
    return th, cost, tt

ratios_A = [0.2, 0.4, 0.6, 0.8, 1.0, 1.5, 2.0] if not QUICK else [0.4, 0.8, 1.5]
rows = []
SEP_TOL = 0.25        # |tau2_hat - tau1_hat| below this (in sigma) = "merged" solution
for init in ["uniform", "near", "perturbed"]:
    for rt in ratios_A:
        d = rt*KAPPA*1.0           # rt = d / H_NWCS  (H_NWCS = kappa*sigma, sigma=1)
        th, cost, tt = adam_multistart(d, init=init, n_starts=60 if QUICK else 200, steps=600 if QUICK else 1500)
        gap = th[:, 1] - th[:, 0]                                  # >0: same ordering as truth (w1 on the earlier return)
        true_b = gap > SEP_TOL; swap_b = gap < -SEP_TOL; merged = ~true_b & ~swap_b
        best = cost.min()
        rows.append(dict(init=init, d_over_H=rt, d=round(d, 3), frac_true=true_b.mean(), frac_swap=swap_b.mean(), frac_merged=merged.mean(),
                         swap_cost_excess=(cost[swap_b].mean()-best) if swap_b.any() else np.nan))
tabA = pd.DataFrame(rows).round(3); print(tabA.to_string(index=False))
print("\nPaper: onset of a stable non-swap basin at d/sigma_pulse = 0.6-0.8 (predicted threshold ratio 1.0); swap basin captured 1-8% of restarts.")
print("The paper does not say which initialisation was used (three are shown). 'swap' = fitted order flipped (tau1_hat > tau2_hat);")
print("'swap_cost_excess' = extra cost of that basin over the best start. If no init reproduces 1-8%, say so in the appendix rather than quoting it.")
# "onset": smallest d/H at which the swap configuration is a statistically distinguishable local minimum (extra NLL >= 3, ~2.4 sigma).
# This is MY reading of the paper's "onset of a stable non-swap basin at 0.6-0.8" -- the paper does not define the criterion.
for _i in tabA.init.unique():
    _s = tabA[(tabA.init == _i) & tabA.swap_cost_excess.notna() & (tabA.swap_cost_excess >= 3.0)]
    print(f"init={_i:>9}: swap basin distinguishable (delta-NLL >= 3) from d/H = {_s.d_over_H.min() if len(_s) else 'never'}")
tabA.to_csv(f"{OUT}/appendixA_swap_basin.csv", index=False)

# oblique-incidence consistency check (theta = 35 deg): pulse broadens to sigma/cos(theta); ratio measured against H_NWCS/cos(theta).
# Because the problem is scale-free, this is a code-consistency check of the 1/cos(theta) scaling, not an independent physical test.
th_obl = np.deg2rad(35); s_obl = 1.0/np.cos(th_obl)
for rt in ([0.8] if QUICK else [0.6, 0.8, 1.0]):
    d = rt*KAPPA*s_obl
    th, cost, tt = adam_multistart(d, sigma=s_obl, init="near", n_starts=60 if QUICK else 200, steps=600 if QUICK else 1500)
    g_ = th[:, 1] - th[:, 0]
    print(f"oblique 35deg  d/H={rt}: frac_true={np.mean(g_ > SEP_TOL*s_obl):.2f}  frac_swap={np.mean(g_ < -SEP_TOL*s_obl):.2f}")

## 8. Real GEDI L2B validation  (Sec. 3, Table 1, App. B)
*Source:* `Untitled22.ipynb` cells 0, 2, 7, 8, 17–24 — **VERBATIM** (needs Earthdata credentials, ~21–25 GEDI granules, Copernicus DEM tiles, GPU).
Cells 9–16 and 25–31 of that notebook (the "IGMC" r̂ analysis) are **not** included: they are not in the paper, and cells 9–16 ran on a *placeholder* dataframe (cell 10 says "REPLACE this cell").
Then **GENERATED** cells compute the Table-1 rows that no notebook produced (roughness r, geolocation r), and re-do the "3–6×" comparison without normalising by the binning variable.

In [ ]:
import getpass, os
username = input("Earthdata username: ")
password = getpass.getpass("Earthdata password: ")
with open(os.path.expanduser("~/.netrc"), "w") as f:
    f.write(f"machine urs.earthdata.nasa.gov\nlogin {username}\npassword {password}\n")
os.chmod(os.path.expanduser("~/.netrc"), 0o600)
print("Credentials saved.")

In [ ]:
import requests, os

def gedi_finder_v2(short_name, version, bbox):
    cmr_url = "https://cmr.earthdata.nasa.gov/search/granules.json"
    params = {"provider":"LPCLOUD","short_name":short_name,"version":version,
              "bounding_box":bbox,"page_size":2000}
    return requests.get(cmr_url, params=params, timeout=60).json().get('feed',{}).get('entry',[])

entries = gedi_finder_v2("GEDI02_B", "002", "-105.5,39.0,-104.5,40.0")
print(f"Found {len(entries)} granules")

download_dir = "gedi_l2b_colorado"
os.makedirs(download_dir, exist_ok=True)
session = requests.Session()

n_to_download = 25
for e in entries[:n_to_download]:
    data_url = next((l['href'] for l in e.get('links',[]) if l.get('href','').endswith('.h5')), None)
    if not data_url: continue
    fname_only = data_url.split('/')[-1]
    fname = os.path.join(download_dir, fname_only)
    if os.path.exists(fname): continue
    r = session.get(data_url, timeout=300)
    if r.status_code == 200:
        with open(fname, 'wb') as f: f.write(r.content)
        print(f"{fname_only}: {len(r.content)/1e6:.1f} MB")
    else:
        print(f"FAILED ({r.status_code}): {fname_only}")

print(f"\nTotal files: {len(os.listdir(download_dir))}")

In [ ]:
import h5py, numpy as np, pandas as pd

bbox_lon_min, bbox_lon_max = -105.5, -104.5
bbox_lat_min, bbox_lat_max = 39.0, 40.0
beam_names = ['BEAM0000','BEAM0001','BEAM0010','BEAM0011',
              'BEAM0101','BEAM0110','BEAM1000','BEAM1011']

all_rows = []
for fname in os.listdir(download_dir):
    fpath = os.path.join(download_dir, fname)
    try:
        with h5py.File(fpath, 'r') as f:
            for beam in beam_names:
                if beam not in f: continue
                try:
                    lat = f[f'{beam}/geolocation/latitude_bin0'][:]
                    lon = f[f'{beam}/geolocation/longitude_bin0'][:]
                    elev = f[f'{beam}/geolocation/elev_lowestmode'][:]
                    rh100 = f[f'{beam}/rh100'][:]
                    cover = f[f'{beam}/cover'][:]
                    pai = f[f'{beam}/pai'][:]
                    qflag = f[f'{beam}/l2b_quality_flag'][:]
                    beam_elev_rad = f[f'{beam}/geolocation/local_beam_elevation'][:]
                except KeyError:
                    continue
                mask = ((lon>=bbox_lon_min)&(lon<=bbox_lon_max)&
                        (lat>=bbox_lat_min)&(lat<=bbox_lat_max)&(qflag==1))
                if mask.sum()==0: continue

                offnadir_rad = (np.pi/2) - beam_elev_rad[mask]
                cos_theta = np.sin(beam_elev_rad[mask])  # = cos(offnadir)

                all_rows.append(pd.DataFrame({
                    'lon':lon[mask],'lat':lat[mask],'elev':elev[mask],
                    'rh100':rh100[mask],'cover':cover[mask],'pai':pai[mask],
                    'offnadir_deg': np.degrees(offnadir_rad),
                    'cos_theta': cos_theta,
                    'file':fname
                }))
    except Exception as ex:
        print(f"Skipping {fname}: {ex}")

gedi_df_angle = pd.concat(all_rows, ignore_index=True)
gedi_clean_angle = gedi_df_angle[(gedi_df_angle['pai']>-100)&(gedi_df_angle['pai']<20)&
                                   (gedi_df_angle['rh100']>0)&(gedi_df_angle['rh100']<5000)&
                                   (gedi_df_angle['cover']>=0)&(gedi_df_angle['cover']<=1)].copy()
low_veg_angle = gedi_clean_angle[gedi_clean_angle['cover']<0.1].copy()

print(f"Total low-veg shots: {len(low_veg_angle):,}")
print(f"\nOff-nadir angle (degrees) across ALL {low_veg_angle['file'].nunique()} files:")
print(low_veg_angle['offnadir_deg'].describe())
print(f"\ncos(theta) stats:")
print(low_veg_angle['cos_theta'].describe())

In [ ]:
# Quantify the maximum possible impact on H_NWCS from this correction
cos_theta_min = low_veg_angle['cos_theta'].min()
h_nwcs_approx = 1.0  # our previous cos_theta_approx=1.0 assumption
h_nwcs_correction_factor = 1.0 / cos_theta_min

print(f"Maximum possible H_NWCS correction factor: {h_nwcs_correction_factor:.5f}")
print(f"i.e. at most a {100*(h_nwcs_correction_factor-1):.3f}% change in H_NWCS "
      f"for the single most-oblique shot in the dataset")
print(f"Median-shot correction: {1/low_veg_angle['cos_theta'].median():.6f} "
      f"({100*(1/low_veg_angle['cos_theta'].median()-1):.4f}% change)")

In [ ]:
import h5py, numpy as np, pandas as pd

bbox_lon_min, bbox_lon_max = -105.5, -104.5
bbox_lat_min, bbox_lat_max = 39.0, 40.0
beam_names = ['BEAM0000','BEAM0001','BEAM0010','BEAM0011',
              'BEAM0101','BEAM0110','BEAM1000','BEAM1011']

all_rows = []
for fname in os.listdir(download_dir):
    fpath = os.path.join(download_dir, fname)
    try:
        with h5py.File(fpath, 'r') as f:
            for beam in beam_names:
                if beam not in f: continue
                try:
                    lat = f[f'{beam}/geolocation/latitude_bin0'][:]
                    lon = f[f'{beam}/geolocation/longitude_bin0'][:]
                    elev = f[f'{beam}/geolocation/elev_lowestmode'][:]
                    rh100 = f[f'{beam}/rh100'][:]
                    cover = f[f'{beam}/cover'][:]
                    pai = f[f'{beam}/pai'][:]
                    qflag = f[f'{beam}/l2b_quality_flag'][:]
                except KeyError:
                    continue
                mask = ((lon>=bbox_lon_min)&(lon<=bbox_lon_max)&
                        (lat>=bbox_lat_min)&(lat<=bbox_lat_max)&(qflag==1))
                if mask.sum()==0: continue
                all_rows.append(pd.DataFrame({'lon':lon[mask],'lat':lat[mask],'elev':elev[mask],
                                                'rh100':rh100[mask],'cover':cover[mask],'pai':pai[mask]}))
    except Exception as ex:
        print(f"Skipping {fname}: {ex}")

gedi_df = pd.concat(all_rows, ignore_index=True)
gedi_clean = gedi_df[(gedi_df['pai']>-100)&(gedi_df['pai']<20)&
                      (gedi_df['rh100']>0)&(gedi_df['rh100']<5000)&
                      (gedi_df['cover']>=0)&(gedi_df['cover']<=1)].copy()
low_veg = gedi_clean[gedi_clean['cover']<0.1].copy()
print(f"Low-vegetation shots: {len(low_veg):,}")

In [ ]:
import rasterio
from rasterio.merge import merge

tiles_needed = ["Copernicus_DSM_COG_10_N39_00_W105_00_DEM",
                "Copernicus_DSM_COG_10_N39_00_W106_00_DEM",
                "Copernicus_DSM_COG_10_N40_00_W105_00_DEM",
                "Copernicus_DSM_COG_10_N40_00_W106_00_DEM"]
base_url = "https://copernicus-dem-30m.s3.amazonaws.com/{t}/{t}.tif"
os.makedirs("copernicus_dem", exist_ok=True)

for tile in tiles_needed:
    fname = f"copernicus_dem/{tile}.tif"
    if os.path.exists(fname): continue
    r = requests.get(base_url.format(t=tile), timeout=120)
    with open(fname, 'wb') as f: f.write(r.content)

src_files = [rasterio.open(f"copernicus_dem/{t}.tif") for t in tiles_needed]
mosaic, transform = merge(src_files)
mosaic = mosaic[0]
for s in src_files: s.close()
print(f"Mosaic: {mosaic.shape}")

In [ ]:
def latlon_to_pixel(lat, lon, transform):
    col, row = ~transform * (lon, lat)
    return int(round(row)), int(round(col))

rough = []
for lon, lat in zip(low_veg['lon'].values, low_veg['lat'].values):
    r, c = latlon_to_pixel(lat, lon, transform)
    r0,r1 = max(0,r-2), min(mosaic.shape[0], r+3)
    c0,c1 = max(0,c-2), min(mosaic.shape[1], c+3)
    w = mosaic[r0:r1, c0:c1]; w = w[w>-9000]
    rough.append(w.std() if len(w)>3 else np.nan)
low_veg['dem_roughness_m'] = rough
low_veg = low_veg.dropna(subset=['dem_roughness_m']).reset_index(drop=True)
print(f"Final dataset: {len(low_veg):,} shots")

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(0); np.random.seed(0)

lon_c, lat_c = low_veg['lon'].mean(), low_veg['lat'].mean()
lon_s = (low_veg['lon'].max()-low_veg['lon'].min())/2
lat_s = (low_veg['lat'].max()-low_veg['lat'].min())/2
X_np = np.stack([(low_veg['lon'].values-lon_c)/lon_s, (low_veg['lat'].values-lat_c)/lat_s], axis=1)

n = len(low_veg)
perm = np.random.permutation(n)
n_train = int(0.8*n)
train_mask = np.zeros(n, dtype=bool); train_mask[perm[:n_train]] = True
holdout_mask = ~train_mask

h_mean = low_veg.loc[train_mask,'elev'].mean()
Y_h = torch.tensor(low_veg['elev'].values - h_mean, dtype=torch.float32, device=device)
Y_sigma = torch.tensor(low_veg['rh100'].values/100.0, dtype=torch.float32, device=device)
X = torch.tensor(X_np, dtype=torch.float32, device=device)

X_train, Yh_train, Ys_train = X[train_mask], Y_h[train_mask], Y_sigma[train_mask]
X_holdout, Yh_holdout = X[holdout_mask], Y_h[holdout_mask]
print(f"Train: {train_mask.sum():,}, Holdout: {holdout_mask.sum():,}")

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, n_freqs=6):
        super().__init__(); self.freqs = 2.0**torch.arange(n_freqs)
    def forward(self, x):
        freqs = self.freqs.to(x.device)
        xp = (x[...,None]*freqs[None,None,:]*np.pi).reshape(x.shape[0],-1)
        return torch.cat([x, torch.sin(xp), torch.cos(xp)], dim=-1)

class NeuralField(nn.Module):
    def __init__(self, n_freqs=6, hidden=192, n_layers=3):
        super().__init__()
        self.pe = PositionalEncoding(n_freqs)
        in_dim = 2 + 2*2*n_freqs
        layers = [nn.Linear(in_dim,hidden), nn.ReLU()]
        for _ in range(n_layers-1): layers += [nn.Linear(hidden,hidden), nn.ReLU()]
        self.trunk = nn.Sequential(*layers)
        self.head_h = nn.Linear(hidden,1); self.head_sigma = nn.Linear(hidden,1)
    def forward(self, x):
        feat = self.trunk(self.pe(x))
        return self.head_h(feat).squeeze(-1), self.head_sigma(feat).squeeze(-1)

model = NeuralField().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
h_std, sigma_std = Yh_train.std(), Ys_train.std()

for step in range(4000):
    idx = torch.randint(0, len(X_train), (4096,), device=device)
    xb, yhb, ysb = X_train[idx], Yh_train[idx], Ys_train[idx]
    optimizer.zero_grad()
    h_pred, sigma_pred = model(xb)
    loss = (F.mse_loss(h_pred,yhb)/(h_std**2+1e-8) + F.mse_loss(sigma_pred,ysb)/(sigma_std**2+1e-8))
    loss.backward(); optimizer.step()

model.eval()
with torch.no_grad():
    h_pred_holdout, _ = model(X_holdout)

In [ ]:
from scipy.spatial import cKDTree

train_xy = X_train.cpu().numpy()
tree = cKDTree(train_xy)
holdout_xy = X_holdout.cpu().numpy()
dists, idxs = tree.query(holdout_xy, k=8)
weights = 1/(dists+1e-6); weights /= weights.sum(axis=1,keepdims=True)
idw_pred = (weights * Yh_train.cpu().numpy()[idxs]).sum(axis=1)

In [ ]:
from scipy import stats
import pandas as pd

# H_NWCS: kappa = 2*sqrt(ln2) (FWHM convention) -- explicitly disclosed, see Sec 4.1
transmit_fwhm_ns = 15.0
sigma_pulse_ns = transmit_fwhm_ns/(2*np.sqrt(np.log(2)))
c_m_per_ns = 0.299792458
kappa = 2*np.sqrt(np.log(2))
H_NWCS_m = kappa * sigma_pulse_ns * c_m_per_ns / 2  # near-nadir approx, verified negligible (0.139%)

holdout_roughness = low_veg.loc[holdout_mask, 'dem_roughness_m'].values
ratio = holdout_roughness / H_NWCS_m

dists_to_train, _ = tree.query(holdout_xy, k=1)
corr_confound, _ = stats.pearsonr(ratio, dists_to_train)
print(f"Distance-ratio confound check: r={corr_confound:.3f} (should be weak)")

true_h = Yh_holdout.cpu().numpy()
err_field = np.abs(h_pred_holdout.cpu().numpy() - true_h)
err_idw = np.abs(idw_pred - true_h)

# Normalize by local roughness scale (removes trivial variance-scaling confound)
rel_err_field = err_field / (ratio * H_NWCS_m + 1e-6)
rel_err_idw = err_idw / (ratio * H_NWCS_m + 1e-6)

res = pd.DataFrame({'ratio':ratio, 'rel_err_field':rel_err_field, 'rel_err_idw':rel_err_idw})
bins = [0,0.5,1,2,5,np.inf]; labels=['<0.5','0.5-1','1-2','2-5','>5']
res['bin'] = pd.cut(res['ratio'], bins=bins, labels=labels)

rel_rmse_field, rel_rmse_idw = [], []
for b in labels:
    sub = res[res['bin']==b]
    rel_rmse_field.append(np.sqrt((sub['rel_err_field']**2).mean()))
    rel_rmse_idw.append(np.sqrt((sub['rel_err_idw']**2).mean()))
    print(f"{b:>6}: n={len(sub):>5}  RelRMSE_field={rel_rmse_field[-1]:.3f}  RelRMSE_idw={rel_rmse_idw[-1]:.3f}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({'font.family':'serif','font.size':10,'savefig.dpi':300,'savefig.bbox':'tight'})

fig, ax = plt.subplots(figsize=(3.4, 2.6))
bin_mid = [0,1,2,3,4]
ax.plot(bin_mid, rel_rmse_field, 'o-', color='#1f77b4', markersize=5, linewidth=1.3, label='Neural Field')
ax.plot(bin_mid, rel_rmse_idw, 's-', color='#ff7f0e', markersize=5, linewidth=1.3, label='IDW')
ax.axvline(1.5, color='#d62728', linestyle=':', linewidth=1.0, alpha=0.7)
ax.set_yscale('log')
ax.set_xticks(bin_mid); ax.set_xticklabels(labels)
ax.set_xlabel(r'$\Delta h / H_{NWCS}$ bin')
ax.set_ylabel('Relative RMSE\n(error / local roughness)')
ax.legend(loc='upper right', frameon=False, fontsize=7.5)
plt.tight_layout()
plt.savefig('fig3_gedi_headline.pdf')
plt.savefig('fig3_gedi_headline.png')
plt.show()

In [ ]:
# ================= GEDI real-data validation: the Table-1 rows that no notebook computed =================
# Generated. Uses the objects created by the copied cells above: low_veg (shots, with dem_roughness_m), mosaic/transform (Copernicus DEM).
from scipy import stats
# (1) roughness vs rh100   -> paper: r = 0.612, p<0.001, N = 7,894
m_ = low_veg[["rh100", "dem_roughness_m"]].dropna()
r_all, p_all = stats.pearsonr(m_.rh100, m_.dem_roughness_m)
r_ho, p_ho = stats.pearsonr(low_veg.loc[holdout_mask, "rh100"], low_veg.loc[holdout_mask, "dem_roughness_m"])
print(f"roughness corr (all {len(m_):,} shots): r={r_all:.3f}, p={p_all:.1e};  holdout only (N={int(holdout_mask.sum()):,}): r={r_ho:.3f}")
print("paper Table 1: r=0.612, N=7,894  (the neural-field holdout in the original run had N=7,886 -> check which subset the paper used)")

# (2) geolocation: GEDI footprint elevation vs Copernicus DEM elevation -> paper: r = 1.000 with a constant -16 m offset
def dem_at(lat, lon):
    col, row = ~transform * (lon, lat); r = int(round(row)); c = int(round(col))
    return mosaic[r, c] if (0 <= r < mosaic.shape[0] and 0 <= c < mosaic.shape[1]) else np.nan
sub = low_veg.sample(min(5000, len(low_veg)), random_state=0)
dem_z = np.array([dem_at(a, b) for a, b in zip(sub.lat, sub.lon)])
ok = np.isfinite(dem_z) & (dem_z > -9000)
r_geo = np.corrcoef(sub.elev.values[ok], dem_z[ok])[0, 1]; off = np.median(sub.elev.values[ok] - dem_z[ok])
print(f"elevation corr vs Copernicus DEM: r={r_geo:.4f}; median offset (GEDI - DEM) = {off:.1f} m;  offset std = {np.std(sub.elev.values[ok]-dem_z[ok]):.1f} m")
print("paper Table 1: r=1.000, -16 m datum offset.  NOTE: r~1 over a range of ~2 km of terrain is a weak test of geolocation; "
      "it cannot detect ~10 m horizontal error. A horizontal-shift scan (correlation vs. DEM shifted by +-30 m) is more informative:")
best = []
for dx in range(-30, 31, 10):
    dz = np.array([dem_at(a, b + dx/ (111320*np.cos(np.deg2rad(a)))) for a, b in zip(sub.lat, sub.lon)])
    g = np.isfinite(dz) & (dz > -9000); best.append((dx, np.corrcoef(sub.elev.values[g], dz[g])[0, 1]))
print("dx(m) -> corr:", [(d_, round(c_, 5)) for d_, c_ in best])

# (3) the "3-6x error reduction" -- recomputed WITHOUT dividing by roughness.
# In the copied cell the plotted metric is RelRMSE = |err| / roughness, and the bins are roughness / H_NWCS. The denominator is the binning
# variable itself, so low bins are inflated mechanically. Report raw errors per bin as well:
err_f = np.abs(h_pred_holdout.cpu().numpy() - Yh_holdout.cpu().numpy()); err_i = np.abs(idw_pred - Yh_holdout.cpu().numpy())
hold_rough = low_veg.loc[holdout_mask, "dem_roughness_m"].values
H_NWCS_m = KAPPA*(15.0/(2*np.sqrt(np.log(2))))*0.299792458/2
ratio_ = hold_rough/H_NWCS_m
dfb = pd.DataFrame(dict(ratio=ratio_, err_field=err_f, err_idw=err_i))
dfb["bin"] = pd.cut(dfb.ratio, [0, 0.5, 1, 2, 5, np.inf], labels=["<0.5", "0.5-1", "1-2", "2-5", ">5"])
g_ = dfb.groupby("bin", observed=True).agg(n=("ratio", "size"), RMSE_field=("err_field", lambda a: np.sqrt((a**2).mean())), RMSE_idw=("err_idw", lambda a: np.sqrt((a**2).mean())))
g_["field_over_idw"] = g_.RMSE_field/g_.RMSE_idw
print("\nRaw (un-normalised) RMSE per roughness bin (metres):"); print(g_.round(3).to_string())
print("If the raw RMSE does not fall 3-6x from the lowest to highest bin, the headline 'transition' in Table 1 is a normalisation effect.")
g_.to_csv(f"{OUT}/gedi_raw_rmse_by_bin.csv")

## 9. Independent airborne-LiDAR validation of ZZHE  (Appendix G, Table 6, Fig. 2)
*Source:* `algorithm_ppr (2).ipynb` cells 1–17, 21–24 — **VERBATIM** (USGS 3DEP download; ~300 MB). Skipped: cell 18/19 (fast-pass & 20-footprint pointwise test) and cell 20.
`algorithm_ppr (1)` is identical except for one extra cell that crashes (`NameError: clean_footprints`). `rng_zzb` (defined in the skipped cells 18/19) is re-defined with the same seed.

In [ ]:
# !pip install -q laspy[lazrs] pyproj requests scipy

In [ ]:
# ============================================================
# Cell 1 (REPLACED) — Query The National Map (TNM) API for LAZ tiles
# covering our bbox. No PDAL needed.
# ============================================================
import requests

BBOX_LON_MIN, BBOX_LAT_MIN, BBOX_LON_MAX, BBOX_LAT_MAX = -105.5, 39.0, -104.5, 40.0

tnm_url = "https://tnmaccess.nationalmap.gov/api/v1/products"
params = {
    "bbox": f"{BBOX_LON_MIN},{BBOX_LAT_MIN},{BBOX_LON_MAX},{BBOX_LAT_MAX}",
    "datasets": "Lidar Point Cloud (LPC)",
    "outputFormat": "JSON",
    "max": 20
}
resp = requests.get(tnm_url, params=params, timeout=60).json()
items = resp.get('items', [])
print(f"Found {len(items)} LAZ tile(s) covering the ROI")
for it in items[:5]:
    print(f"  {it.get('title')}  size={it.get('sizeInBytes', 0)/1e6:.1f} MB  url={it.get('downloadURL')}")

In [ ]:
# ============================================================
# Cell 2 (REPLACED) — Download a small number of tiles directly
# (streamed to disk, memory-safe, same pattern as our GEDI downloads)
# ============================================================
import os

download_dir_als = "colorado_als_tiles"
os.makedirs(download_dir_als, exist_ok=True)

n_tiles_to_download = 3   # start small; each tile can be several hundred MB
CHUNK = 1024 * 1024

downloaded_files = []
for it in items[:n_tiles_to_download]:
    url = it.get('downloadURL')
    if not url or not url.endswith(('.laz', '.las')):
        continue
    fname = os.path.join(download_dir_als, url.split('/')[-1])
    if os.path.exists(fname):
        downloaded_files.append(fname)
        print(f"Already have {fname}")
        continue
    with requests.get(url, stream=True, timeout=1200) as r:
        if r.status_code != 200:
            print(f"FAILED ({r.status_code}): {url}")
            continue
        total = 0
        with open(fname, 'wb') as f:
            for chunk in r.iter_content(chunk_size=CHUNK):
                if chunk:
                    f.write(chunk)
                    total += len(chunk)
        print(f"{fname.split('/')[-1]}: {total/1e6:.1f} MB")
        downloaded_files.append(fname)

print(f"\nTotal tiles downloaded: {len(downloaded_files)}")

In [ ]:
# ============================================================
# Cell 3 (with CRS fallback) — Read tiles with laspy
# ============================================================
import laspy
import numpy as np

all_x, all_y, all_z = [], [], []

for fpath in downloaded_files:
    try:
        las = laspy.read(fpath)
    except Exception as ex:
        print(f"Skipping {fpath}: {ex}")
        continue

    try:
        crs = las.header.parse_crs()
    except Exception as ex:
        crs = None
        print(f"  CRS auto-parse failed ({ex}); will need manual EPSG lookup from tile metadata/VLRs")

    print(f"{fpath}: {len(las.points):,} points, CRS={crs}")

    all_x.append(np.asarray(las.x))
    all_y.append(np.asarray(las.y))
    all_z.append(np.asarray(las.z))

print(f"\nTotal points loaded (before CRS check/reprojection): "
      f"{sum(len(x) for x in all_x):,}")

# Print raw coordinate ranges as a sanity check regardless of CRS metadata --
# if these numbers look like ~400000-500000 (easting) and ~4400000-4500000
# (northing), that's UTM Zone 13N (very likely for Colorado), not lon/lat.
print(f"\nRaw X range: [{all_x[0].min():.1f}, {all_x[0].max():.1f}]")
print(f"Raw Y range: [{all_y[0].min():.1f}, {all_y[0].max():.1f}]")

In [ ]:
# ============================================================
# Cell 3b — Reproject UTM (EPSG:6342) points to lon/lat (EPSG:4326)
# so they line up with our footprint-candidate generation logic.
# No PDAL needed -- pure pyproj on numpy arrays.
# ============================================================
from pyproj import Transformer
import numpy as np
all_x_utm = np.concatenate(all_x)
all_y_utm = np.concatenate(all_y)
all_z_m = np.concatenate(all_z)   # NAVD88 height, meters -- already usable directly

transformer_utm_to_wgs84 = Transformer.from_crs("EPSG:6342", "EPSG:4326", always_xy=True)
all_lon, all_lat = transformer_utm_to_wgs84.transform(all_x_utm, all_y_utm)

print(f"Total points: {len(all_x_utm):,}")
print(f"Lon range: [{all_lon.min():.5f}, {all_lon.max():.5f}]")
print(f"Lat range: [{all_lat.min():.5f}, {all_lat.max():.5f}]")
print(f"Z range: [{all_z_m.min():.1f}, {all_z_m.max():.1f}] m")

# Sanity: this tile's footprint should sit INSIDE our original ROI bbox
print(f"\nOur ROI: lon=[{BBOX_LON_MIN},{BBOX_LON_MAX}]  lat=[{BBOX_LAT_MIN},{BBOX_LAT_MAX}]")

In [ ]:
# ============================================================
# Cell 4 (UPDATED) — Sample candidates within the ACTUAL loaded tile
# extent, not the full original ROI (since we only have ~1 km^2 loaded)
# ============================================================
rng = np.random.default_rng(0)
FOOTPRINT_DIAM_M = 25.0

n_candidates = 800
cand_x = rng.uniform(all_x_utm.min() + 20, all_x_utm.max() - 20, n_candidates)
cand_y = rng.uniform(all_y_utm.min() + 20, all_y_utm.max() - 20, n_candidates)

cand_lon, cand_lat = transformer_utm_to_wgs84.transform(cand_x, cand_y)
print(f"{n_candidates} candidate footprint centers generated within loaded tile extent")

In [ ]:
# ============================================================
# Cell 5 — Extract each footprint's point cloud + ground-truth d_true
# via bimodal peak-finding on the vertical (Z) distribution.
# ============================================================
from scipy.signal import find_peaks
from scipy.stats import gaussian_kde

def extract_footprint_points(cx, cy, x_arr, y_arr, z_arr, radius_m):
    dist2 = (x_arr - cx) ** 2 + (y_arr - cy) ** 2
    mask = dist2 <= radius_m ** 2
    return z_arr[mask]

def ground_truth_two_echo(z_vals, min_points=50, bandwidth=0.5):
    if len(z_vals) < min_points:
        return None, False
    z_grid = np.linspace(z_vals.min(), z_vals.max(), 300)
    try:
        kde = gaussian_kde(z_vals, bw_method=bandwidth / z_vals.std() if z_vals.std() > 0 else 0.3)
    except Exception:
        return None, False
    density = kde(z_grid)
    peaks, props = find_peaks(density, prominence=density.max() * 0.15)
    if len(peaks) < 2:
        return None, False
    top2 = peaks[np.argsort(props['prominences'])[-2:]]
    top2_sorted = np.sort(z_grid[top2])
    d_true = top2_sorted[1] - top2_sorted[0]
    return d_true, True

FOOTPRINT_RADIUS_M = FOOTPRINT_DIAM_M / 2.0
footprints = []
n_points_per_footprint = []

for cx, cy, clon, clat in zip(cand_x, cand_y, cand_lon, cand_lat):
    z_vals = extract_footprint_points(cx, cy, all_x_utm, all_y_utm, all_z_m, FOOTPRINT_RADIUS_M)
    n_points_per_footprint.append(len(z_vals))
    d_true, valid = ground_truth_two_echo(z_vals)
    if valid and 0.3 < d_true < 30.0:
        footprints.append({'lon': clon, 'lat': clat, 'x': cx, 'y': cy,
                            'z_vals': z_vals, 'd_true': d_true})

n_points_per_footprint = np.array(n_points_per_footprint)
print(f"Points per footprint: median={np.median(n_points_per_footprint):.0f}, "
      f"mean={n_points_per_footprint.mean():.0f}, "
      f"min={n_points_per_footprint.min()}, max={n_points_per_footprint.max()}")
print(f"\nValid two-echo footprints found: {len(footprints)} / {n_candidates}")

if footprints:
    d_trues = np.array([f['d_true'] for f in footprints])
    print(f"d_true range: [{d_trues.min():.2f}, {d_trues.max():.2f}] m, "
          f"mean={d_trues.mean():.2f}, median={np.median(d_trues):.2f}")

In [ ]:
# ============================================================
# Cell 6 — Synthesize a realistic waveform per footprint (Blair &
# Hofton 1999 convolution principle): bin the point cloud vertically,
# convolve with the real GEDI system pulse, add realistic noise.
# ============================================================
GEDI_PULSE_FWHM_NS = 15.0
C_M_PER_NS = 0.299792458
SIGMA_PULSE_NS = GEDI_PULSE_FWHM_NS / (2 * np.sqrt(np.log(2)))
SIGMA_PULSE_M = SIGMA_PULSE_NS * C_M_PER_NS / 2.0
print(f"sigma_pulse in meters (one-way): {SIGMA_PULSE_M:.4f}")

REAL_SIGMA_N = 0.15   # placeholder amplitude-noise level; refine later against
                       # documented GEDI noise if L1B noise_stddev is pulled in

def synthesize_waveform(z_vals, sigma_pulse_m, sigma_n, z_grid_res=0.15, pad_m=10.0, rng=None):
    z_min, z_max = z_vals.min() - pad_m, z_vals.max() + pad_m
    z_grid = np.arange(z_min, z_max, z_grid_res)
    counts, edges = np.histogram(z_vals, bins=z_grid)
    centers = 0.5 * (edges[:-1] + edges[1:])
    density = counts.astype(np.float64)
    if density.sum() > 0:
        density /= density.max()

    kernel_half_width = max(1, int(4 * sigma_pulse_m / z_grid_res))
    k_grid = np.arange(-kernel_half_width, kernel_half_width + 1) * z_grid_res
    kernel = np.exp(-k_grid ** 2 / (2 * sigma_pulse_m ** 2))
    kernel /= kernel.sum()
    waveform_clean = np.convolve(density, kernel, mode='same')

    waveform_noisy = waveform_clean + rng.normal(0, sigma_n, size=waveform_clean.shape)
    return centers, waveform_noisy, waveform_clean

rng = np.random.default_rng(42)
for f in footprints:
    centers, wf_noisy, wf_clean = synthesize_waveform(f['z_vals'], SIGMA_PULSE_M, REAL_SIGMA_N, rng=rng)
    f['t_grid'] = centers
    f['waveform'] = wf_noisy

print(f"Synthesized waveforms for {len(footprints)} footprints")
print(f"Sample waveform length: {len(footprints[0]['t_grid'])} bins")

In [ ]:
# ============================================================
# Cell 7 — Fit two-echo MLE to each synthesized waveform
# ============================================================
from scipy.optimize import minimize

def pulse(t, tau, sigma_pulse):
    return np.exp(-(t - tau) ** 2 / (2 * sigma_pulse ** 2))

def mu_waveform_real(params, t_grid, sigma_pulse):
    c, d, w1, w2 = params
    tau1, tau2 = c - d / 2.0, c + d / 2.0
    return w1 * pulse(t_grid, tau1, sigma_pulse) + w2 * pulse(t_grid, tau2, sigma_pulse)

def neg_log_lik_real(params, wf, t_grid, sigma_pulse, sigma_n):
    mu = mu_waveform_real(params, t_grid, sigma_pulse)
    return 0.5 * np.sum((wf - mu) ** 2) / sigma_n ** 2

def fit_two_echo(f, sigma_pulse, sigma_n, n_restarts=6, rng=None):
    t_grid, wf = f['t_grid'], f['waveform']
    c0_guess = t_grid[np.argmax(wf)]
    best, best_cost = None, np.inf
    for _ in range(n_restarts):
        c0 = c0_guess + rng.normal(0, 2)
        d0 = rng.uniform(0.5, 25)
        w0 = max(wf.max(), 0.1)
        res = minimize(neg_log_lik_real, x0=[c0, d0, w0, w0],
                        args=(wf, t_grid, sigma_pulse, sigma_n), method='Nelder-Mead',
                        options={'maxiter': 2000})
        if res.fun < best_cost:
            best_cost, best = res.fun, res.x
    return best

for f in footprints:
    fit = fit_two_echo(f, SIGMA_PULSE_M, REAL_SIGMA_N, rng=rng)
    f['d_hat'] = abs(fit[1])
    f['sq_err'] = (f['d_hat'] - f['d_true']) ** 2

d_hats = np.array([f['d_hat'] for f in footprints])
sq_errs = np.array([f['sq_err'] for f in footprints])
print(f"Fitting complete over {len(footprints)} footprints.")
print(f"d_hat range: [{d_hats.min():.2f}, {d_hats.max():.2f}]")
print(f"sq_err: median={np.median(sq_errs):.4f}, mean={sq_errs.mean():.4f}")
print("\nSample:")
for f in footprints[:8]:
    print(f"  d_true={f['d_true']:.2f}  d_hat={f['d_hat']:.2f}  sq_err={f['sq_err']:.4f}")

In [ ]:
# ============================================================
# Cell 7a — Checkpoint EVERYTHING we have right now, before doing
# anything else. This alone prevents another 38-min disaster.
# ============================================================
import pickle

with open('footprints_with_fits.pkl', 'wb') as f:
    pickle.dump(footprints, f)
print(f"Checkpointed {len(footprints)} footprints (with d_true, d_hat, sq_err) to disk.")

# Recovery cell, for next time a restart happens -- run this FIRST always:
# import pickle
# with open('footprints_with_fits.pkl', 'rb') as f:
#     footprints = pickle.load(f)
# print(f"Restored {len(footprints)} footprints, no redownload/refit needed")

In [ ]:
# ============================================================
# Cell 7b — Diagnostic: real vs synthetic-style multi-modal structure
# ============================================================
def count_all_peaks(z_vals, min_points=50, bandwidth=0.5, prominence_frac=0.15):
    if len(z_vals) < min_points:
        return None
    z_grid = np.linspace(z_vals.min(), z_vals.max(), 300)
    try:
        kde = gaussian_kde(z_vals, bw_method=bandwidth / z_vals.std() if z_vals.std() > 0 else 0.3)
    except Exception:
        return None
    density = kde(z_grid)
    peaks, props = find_peaks(density, prominence=density.max() * prominence_frac)
    return len(peaks)

for f in footprints:
    f['n_peaks_total'] = count_all_peaks(f['z_vals'])

n_peaks_arr = np.array([f['n_peaks_total'] for f in footprints])
print("Peak-count distribution:")
for k in sorted(set(n_peaks_arr)):
    print(f"  {k} peaks: {(n_peaks_arr==k).sum()} footprints")

sq_errs_by_npeaks = {}
for f in footprints:
    sq_errs_by_npeaks.setdefault(f['n_peaks_total'], []).append(f['sq_err'])
print("\nMedian sq_err by peak-count:")
for k in sorted(sq_errs_by_npeaks):
    v = sq_errs_by_npeaks[k]
    print(f"  {k} peaks (n={len(v)}): median={np.median(v):.3f}")

In [ ]:
# ============================================================
# Cell 7c — Filter to strictly-bimodal footprints only
# This is a pre-registered, model-scope-matched filter (our theory
# is a two-echo model; we test it on genuinely two-echo data), not
# cherry-picking based on which points fit well.
# ============================================================
clean_footprints = [f for f in footprints if f['n_peaks_total'] == 2]
print(f"Clean bimodal subset: {len(clean_footprints)} / {len(footprints)}")

clean_sq_errs = np.array([f['sq_err'] for f in clean_footprints])
print(f"sq_err on clean subset: median={np.median(clean_sq_errs):.4f}, mean={clean_sq_errs.mean():.4f}")

with open('clean_footprints.pkl', 'wb') as f:
    pickle.dump(clean_footprints, f)

In [ ]:
# ============================================================
# Cell 7d — Self-contained ZZB machinery (redefines everything that
# was lost on kernel restart). Run this BEFORE Cell 8, always.
# ============================================================
import numpy as np
from scipy.optimize import minimize

_trapz = getattr(np, "trapezoid", None) or np.trapz

def pulse(t, tau, sigma_pulse):
    return np.exp(-(t - tau) ** 2 / (2 * sigma_pulse ** 2))

def marginal_likelihood_mc(x_waveform, d, c_grid, t_grid, sigma_pulse, sigma_n, w1, w2):
    t = t_grid[:, None]
    c = c_grid[None, :]
    tau1, tau2 = c - d / 2.0, c + d / 2.0
    mu = w1 * pulse(t, tau1, sigma_pulse) + w2 * pulse(t, tau2, sigma_pulse)
    resid = x_waveform[:, None] - mu
    log_lik = -0.5 * np.sum(resid ** 2, axis=0) / sigma_n ** 2
    max_ll = log_lik.max()
    L = np.exp(log_lik - max_ll).mean()
    return L, max_ll

def p_err_direct_mc(d, h, c_center, t_grid, sigma_pulse, sigma_n, w1, w2, c_grid, n_sim_per_hyp, rng):
    n_correct, n_total = 0, 2 * n_sim_per_hyp
    for hyp, d_true in [(0, d), (1, d + h)]:
        for _ in range(n_sim_per_hyp):
            tau1, tau2 = c_center - d_true / 2.0, c_center + d_true / 2.0
            mu_true = w1 * pulse(t_grid, tau1, sigma_pulse) + w2 * pulse(t_grid, tau2, sigma_pulse)
            x = mu_true + rng.normal(0, sigma_n, size=t_grid.shape)
            L0, max0 = marginal_likelihood_mc(x, d, c_grid, t_grid, sigma_pulse, sigma_n, w1, w2)
            L1, max1 = marginal_likelihood_mc(x, d + h, c_grid, t_grid, sigma_pulse, sigma_n, w1, w2)
            decision = 0 if (max0 + np.log(L0 + 1e-300)) >= (max1 + np.log(L1 + 1e-300)) else 1
            if decision == hyp:
                n_correct += 1
    return 1.0 - n_correct / n_total

def zzb_floor(d, c_center, t_grid, sigma_pulse, sigma_n, w1, w2, c_grid, h_grid, n_sim_per_hyp, rng):
    p_errs = np.array([p_err_direct_mc(d, h, c_center, t_grid, sigma_pulse, sigma_n, w1, w2,
                                         c_grid, n_sim_per_hyp, rng) for h in h_grid])
    integrand = h_grid * p_errs
    integrand_vf = np.maximum.accumulate(integrand[::-1])[::-1]
    return 0.5 * _trapz(integrand_vf, h_grid)

print("ZZB machinery redefined and ready.")

In [ ]:
rng_zzb = np.random.default_rng(99)   # same seed as in the skipped cells 18/19 of algorithm_ppr

In [ ]:
# NOTE on the ZZB used for Table 6: the copied 'Cell 10' / 'Cell 13' compute ONE floor per bin at a representative footprint
# with n_sim=150 and a 15-point h-grid (not the 'locked' 800 / 40 used elsewhere), and 'Cell 13' picks the representative footprint with
# abs(d_true - median(errs_enlarged)) -- i.e. it takes the median of the SQUARED ERRORS, not of d_true. The continuous ablation below
# (canonical d-curve, interpolated at every footprint's own d_true) avoids both issues.

In [ ]:
# ============================================================
# Cell 10 (FIXED) — use each bin's OWN representative footprint's
# actual t_grid and c_center (from where its waveform peaks), instead
# of a mismatched global c_center=0.0 and borrowed t_grid.
# ============================================================
bins_edges = [0, 4, 8, 12, 25]
bin_labels = ['<4m', '4-8m', '8-12m', '>12m']

for lo, hi, label in zip(bins_edges[:-1], bins_edges[1:], bin_labels):
    bin_fps = [f for f in clean_footprints if lo <= f['d_true'] < hi]
    if len(bin_fps) < 3:
        print(f"{label}: only {len(bin_fps)} footprints, skipping")
        continue
    errs = np.array([f['sq_err'] for f in bin_fps])
    mean_err, se_err = errs.mean(), errs.std(ddof=1) / np.sqrt(len(errs))

    # pick the bin's median-d_true footprint AS the representative,
    # and use ITS OWN t_grid and fitted c_center -- these are guaranteed
    # to be on the same scale as each other.
    rep_fp = sorted(bin_fps, key=lambda f: abs(f['d_true'] - np.median([x['d_true'] for x in bin_fps])))[0]
    d_rep = rep_fp['d_true']
    t_grid_rep = rep_fp['t_grid']
    c_center = t_grid_rep[np.argmax(rep_fp['waveform'])]
    c_grid_f = np.linspace(c_center - 15, c_center + 15, 30)

    zzb = zzb_floor(d_rep, c_center, t_grid_rep, SIGMA_PULSE_M, REAL_SIGMA_N,
                     1.0, 1.0, c_grid_f, np.geomspace(0.1, 25.0, 15), 150, rng_zzb)

    margin_pass = zzb <= (mean_err - 2 * se_err)
    print(f"{label} (n={len(bin_fps)}, d_rep={d_rep:.2f}, c_center={c_center:.1f}): "
          f"mean_err={mean_err:.3f}±{se_err:.3f}  ZZB={zzb:.4f}  margin_pass={margin_pass}")

In [ ]:
# ============================================================
# Cell 11 — More footprints specifically in the <4m (near-threshold)
# bin. We already have 16; check if more exist in the full 800-candidate
# pool that weren't in our first random draw, or draw fresh candidates
# concentrated in areas likely to have small canopy-gaps.
# ============================================================
near_thresh_bin = [f for f in clean_footprints if f['d_true'] < 4.0]
print(f"Current <4m bin size: {len(near_thresh_bin)}")

# First, check: does the outlier (d_true=3.32, sq_err=105.9) hold up,
# or is it itself a fitting artifact worth inspecting directly?
outlier = [f for f in near_thresh_bin if abs(f['d_true'] - 3.32) < 0.05]
if outlier:
    f = outlier[0]
    print(f"\nOutlier detail: d_true={f['d_true']:.3f}  d_hat={f['d_hat']:.3f}  "
          f"n_peaks_total={f['n_peaks_total']}")

In [ ]:
# ============================================================
# Cell 12 — Draw additional candidates from the loaded tile, this time
# checking more points to enrich the <4m bin specifically.
# ============================================================
rng2 = np.random.default_rng(7)
n_extra_candidates = 1500  # draw more since <4m footprints were a small
                             # fraction of our first 800 draws

extra_x = rng2.uniform(all_x_utm.min() + 20, all_x_utm.max() - 20, n_extra_candidates)
extra_y = rng2.uniform(all_y_utm.min() + 20, all_y_utm.max() - 20, n_extra_candidates)
extra_lon, extra_lat = transformer_utm_to_wgs84.transform(extra_x, extra_y)

extra_footprints = []
for cx, cy, clon, clat in zip(extra_x, extra_y, extra_lon, extra_lat):
    z_vals = extract_footprint_points(cx, cy, all_x_utm, all_y_utm, all_z_m, FOOTPRINT_RADIUS_M)
    d_true, valid = ground_truth_two_echo(z_vals)
    if valid and 0.3 < d_true < 4.0:   # only keep near-threshold ones this time
        extra_footprints.append({'lon': clon, 'lat': clat, 'x': cx, 'y': cy,
                                   'z_vals': z_vals, 'd_true': d_true})

print(f"New near-threshold footprints found: {len(extra_footprints)}")

for f in extra_footprints:
    centers, wf_noisy, wf_clean = synthesize_waveform(f['z_vals'], SIGMA_PULSE_M, REAL_SIGMA_N, rng=rng2)
    f['t_grid'] = centers
    f['waveform'] = wf_noisy
    fit = fit_two_echo(f, SIGMA_PULSE_M, REAL_SIGMA_N, rng=rng2)
    f['d_hat'] = abs(fit[1])
    f['sq_err'] = (f['d_hat'] - f['d_true']) ** 2
    f['n_peaks_total'] = count_all_peaks(f['z_vals'])

extra_clean = [f for f in extra_footprints if f['n_peaks_total'] == 2]
print(f"Clean (bimodal) new footprints: {len(extra_clean)}")

combined_near_thresh = near_thresh_bin + extra_clean
print(f"Combined <4m bin size: {len(combined_near_thresh)}")

with open('combined_near_thresh.pkl', 'wb') as f:
    pickle.dump(combined_near_thresh, f)

In [ ]:
# ============================================================
# Cell 13 — Re-test the <4m bin with the enlarged sample
# ============================================================
errs_enlarged = np.array([f['sq_err'] for f in combined_near_thresh])
mean_err_e, se_err_e = errs_enlarged.mean(), errs_enlarged.std(ddof=1) / np.sqrt(len(errs_enlarged))

rep_fp = sorted(combined_near_thresh, key=lambda f: abs(f['d_true'] - np.median(errs_enlarged)))[0]
d_rep = rep_fp['d_true']
t_grid_rep = rep_fp['t_grid']
c_center = t_grid_rep[np.argmax(rep_fp['waveform'])]
c_grid_f = np.linspace(c_center - 15, c_center + 15, 30)

zzb = zzb_floor(d_rep, c_center, t_grid_rep, SIGMA_PULSE_M, REAL_SIGMA_N,
                 1.0, 1.0, c_grid_f, np.geomspace(0.1, 25.0, 15), 150, rng_zzb)

margin_pass = zzb <= (mean_err_e - 2 * se_err_e)
print(f"<4m enlarged (n={len(combined_near_thresh)}, d_rep={d_rep:.2f}): "
      f"mean_err={mean_err_e:.3f}±{se_err_e:.3f}  ZZB={zzb:.4f}  margin_pass={margin_pass}")

In [ ]:
# ---------- Figure 2b / Table 6 summary from the copied ALS cells ----------
bins_edges = [0, 4, 8, 12, 25]; bin_labels = ["<4 m", "4-8 m", "8-12 m", ">12 m"]
allfp = [f for f in clean_footprints if f["d_true"] >= 4.0] + list(combined_near_thresh)
rows = []
for lo, hi, lab in zip(bins_edges[:-1], bins_edges[1:], bin_labels):
    e = np.array([f["sq_err"] for f in allfp if lo <= f["d_true"] < hi])
    rows.append(dict(bin=lab, n=len(e), mean_err=e.mean(), se=e.std(ddof=1)/np.sqrt(len(e)), median_err=np.median(e)))
t6 = pd.DataFrame(rows); print(t6.round(3).to_string(index=False))
plt.figure(figsize=(4.5, 3.2)); plt.bar(t6.bin, t6.mean_err, yerr=2*t6.se); plt.ylabel("squared error (m^2)"); plt.tight_layout()
plt.savefig(f"{OUT}/fig2_als_validation.png", dpi=200); plt.show()
t6.to_csv(f"{OUT}/table6_als.csv", index=False)
print("Paper: mean err 67.2/22.2/35.9/86.2 m^2; n = 36/82/157/71.  NOTE the median error is far below the mean (heavy tail) -- report both.")

### 9b. Ablations on the ALS result (continuous L_ZZB curve; near-optimal MMSE baseline)
*Source:* `ablation_pipeline (2).ipynb` cells 2–8, 11–14 — **PATCHED** (`QUICK`; checkpoint pickles written first so the cells run on the real footprints instead of the synthetic stand-in).
**The uploaded copy of this notebook ran on a 60-footprint synthetic stand-in** ("No checkpoint found"), so none of its printed numbers are real-data results.

In [ ]:
import pickle
with open('clean_footprints.pkl','wb') as fh: pickle.dump(clean_footprints, fh)
with open('combined_near_thresh.pkl','wb') as fh: pickle.dump(combined_near_thresh, fh)

In [ ]:
# Cell A1 — Load checkpoints, tag enrichment status, build the combined dataset
import pickle, numpy as np

def _load(path):
    try:
        with open(path, 'rb') as fh:
            return pickle.load(fh)
    except FileNotFoundError:
        return None

clean_footprints = _load('clean_footprints.pkl')
combined_near_thresh = _load('combined_near_thresh.pkl')

if clean_footprints is None:
    print("No checkpoint found -- run the main pipeline first. "
          "Falling back to a tiny synthetic stand-in so this notebook still runs.")
    rng0 = np.random.default_rng(0)
    clean_footprints = []
    for i in range(60):
        d_true = float(rng0.uniform(1.5, 22.0))
        clean_footprints.append({
            'lon': -105.0 + i * 1e-4, 'lat': 39.5 + i * 1e-4,
            'd_true': d_true, 'd_hat': d_true + rng0.normal(0, 0.3 + 0.02 * d_true),
        })
    for f in clean_footprints:
        f['sq_err'] = (f['d_hat'] - f['d_true']) ** 2
    combined_near_thresh = None

for f in clean_footprints:
    f.setdefault('enriched', False)

near_thresh_original_ids = {
    (round(f['lon'], 6), round(f['lat'], 6))
    for f in clean_footprints if f['d_true'] < 4.0
}

new_enriched = []
if combined_near_thresh is not None:
    new_enriched = [
        f for f in combined_near_thresh
        if (round(f['lon'], 6), round(f['lat'], 6)) not in near_thresh_original_ids
    ]
    for f in new_enriched:
        f['enriched'] = True

all_footprints = clean_footprints + new_enriched
d_vals = np.array([f['d_true'] for f in all_footprints])
sq_errs = np.array([f['sq_err'] for f in all_footprints])
was_enriched = np.array([f['enriched'] for f in all_footprints])

print(f"Base clean footprints:        {len(clean_footprints)}")
print(f"New enriched (<4m) footprints added: {len(new_enriched)}")
print(f"Combined dataset:             {len(all_footprints)}")
print(f"d_true range: [{d_vals.min():.2f}, {d_vals.max():.2f}] m")


In [ ]:
# Cell A2 — Self-contained ZZB machinery (identical to Cell 7d in the main pipeline)
from scipy.optimize import minimize

_trapz = getattr(np, "trapezoid", None) or np.trapz

def pulse(t, tau, sigma_pulse):
    return np.exp(-(t - tau) ** 2 / (2 * sigma_pulse ** 2))

def marginal_likelihood_mc(x_waveform, d, c_grid, t_grid, sigma_pulse, sigma_n, w1, w2):
    t = t_grid[:, None]
    c = c_grid[None, :]
    tau1, tau2 = c - d / 2.0, c + d / 2.0
    mu = w1 * pulse(t, tau1, sigma_pulse) + w2 * pulse(t, tau2, sigma_pulse)
    resid = x_waveform[:, None] - mu
    log_lik = -0.5 * np.sum(resid ** 2, axis=0) / sigma_n ** 2
    max_ll = log_lik.max()
    L = np.exp(log_lik - max_ll).mean()
    return L, max_ll

def p_err_direct_mc(d, h, c_center, t_grid, sigma_pulse, sigma_n, w1, w2, c_grid, n_sim_per_hyp, rng):
    n_correct, n_total = 0, 2 * n_sim_per_hyp
    for hyp, d_true in [(0, d), (1, d + h)]:
        for _ in range(n_sim_per_hyp):
            tau1, tau2 = c_center - d_true / 2.0, c_center + d_true / 2.0
            mu_true = w1 * pulse(t_grid, tau1, sigma_pulse) + w2 * pulse(t_grid, tau2, sigma_pulse)
            x = mu_true + rng.normal(0, sigma_n, size=t_grid.shape)
            L0, max0 = marginal_likelihood_mc(x, d, c_grid, t_grid, sigma_pulse, sigma_n, w1, w2)
            L1, max1 = marginal_likelihood_mc(x, d + h, c_grid, t_grid, sigma_pulse, sigma_n, w1, w2)
            decision = 0 if (max0 + np.log(L0 + 1e-300)) >= (max1 + np.log(L1 + 1e-300)) else 1
            if decision == hyp:
                n_correct += 1
    return 1.0 - n_correct / n_total

def zzb_floor(d, c_center, t_grid, sigma_pulse, sigma_n, w1, w2, c_grid, h_grid, n_sim_per_hyp, rng):
    p_errs = np.array([p_err_direct_mc(d, h, c_center, t_grid, sigma_pulse, sigma_n, w1, w2,
                                         c_grid, n_sim_per_hyp, rng) for h in h_grid])
    integrand = h_grid * p_errs
    integrand_vf = np.maximum.accumulate(integrand[::-1])[::-1]
    return 0.5 * _trapz(integrand_vf, h_grid)

print("ZZB machinery ready.")


In [ ]:
# Cell A3 — Canonical continuous L_ZZB(d) curve
#
# Key point: the ZZB computation only depends on relative offsets (pulse shape,
# sigma_pulse, sigma_n) -- NOT on the absolute elevation scale. This is exactly
# what the earlier c_center=0.0 bug got wrong: it mixed a placeholder center with
# a REAL-scale t_grid (~2800-3300 m), producing a degenerate, d-independent
# result. Here we deliberately use a small CANONICAL t_grid (centered at 0,
# spanning +-15 m) for every d in the sweep -- this is equivalent to using each
# footprint's own (real-scale) c_center/t_grid, since the physics is
# translation-invariant, but lets us compute ONE shared curve instead of one
# per footprint.

GEDI_PULSE_FWHM_NS = 15.0
C_M_PER_NS = 0.299792458
SIGMA_PULSE_M = (GEDI_PULSE_FWHM_NS / (2 * np.sqrt(np.log(2)))) * C_M_PER_NS / 2.0
REAL_SIGMA_N = 0.15  # known placeholder -- see Open Problems in the dossier; not touched here

canonical_t_grid = np.arange(-15.0, 15.0, 0.15)
canonical_c_center = 0.0
c_grid_canonical = np.linspace(-15, 15, 30)

# Reduced settings for interactive runtime; bump to (40, 800) to match the
# main pipeline's locked settings before trusting exact numbers.
H_GRID_ABLATION = np.geomspace(0.05, 30.0, 40 if not QUICK else 12)
N_SIM_ABLATION = 800 if not QUICK else 60

d_grid_zzb = np.geomspace(max(d_vals.min(), 0.3), d_vals.max(), 18)
rng_zzb = np.random.default_rng(99)

L_zzb_on_grid = np.array([
    zzb_floor(d, canonical_c_center, canonical_t_grid, SIGMA_PULSE_M, REAL_SIGMA_N,
              1.0, 1.0, c_grid_canonical, H_GRID_ABLATION, N_SIM_ABLATION, rng_zzb)
    for d in d_grid_zzb
])

with open('zzb_curve_ablation.pkl', 'wb') as fh:
    pickle.dump({'d_grid': d_grid_zzb, 'L_zzb': L_zzb_on_grid}, fh)

print("d_grid  ->  L_ZZB(d)")
for d, l in zip(d_grid_zzb, L_zzb_on_grid):
    print(f"  {d:6.2f} m  ->  {l:.5f}")


In [ ]:
# Cell A4 — Interpolate L_ZZB at each footprint's OWN d_true, compute residual
# Interpolation is done in log(d) space since d spans nearly two orders of
# magnitude (2-21 m) and L_ZZB(d) is expected to vary smoothly on a log scale
# near the collapse region.
from numpy import interp

log_d_grid = np.log(d_grid_zzb)
log_d_vals = np.log(np.clip(d_vals, d_grid_zzb.min(), d_grid_zzb.max()))
L_zzb_per_footprint = np.exp(interp(log_d_vals, log_d_grid, np.log(np.clip(L_zzb_on_grid, 1e-12, None))))

residual = sq_errs - L_zzb_per_footprint

print(f"Residual (sq_err - L_ZZB) summary:")
print(f"  min={residual.min():.4f}  median={np.median(residual):.4f}  max={residual.max():.4f}")
n_violations_pointwise = int((residual < 0).sum())
print(f"  pointwise violations (sq_err < L_ZZB, no margin): {n_violations_pointwise} / {len(residual)}")


In [ ]:
# Cell A5 — Continuous regression band (LOESS + bootstrap CI) vs the ZZB floor
import matplotlib.pyplot as plt
try:
    from statsmodels.nonparametric.smoothers_lowess import lowess
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--break-system-packages', '-q', 'statsmodels'])
    from statsmodels.nonparametric.smoothers_lowess import lowess

d_plot_grid = np.geomspace(d_vals.min(), d_vals.max(), 200)

def bootstrap_loess_band(d_arr, y_arr, xvals, n_boot=400, frac=0.5, seed=0):
    rng_b = np.random.default_rng(seed)
    n = len(d_arr)
    boots = np.zeros((n_boot, len(xvals)))
    for b in range(n_boot):
        idx = rng_b.integers(0, n, n)
        fit = lowess(y_arr[idx], d_arr[idx], frac=frac, xvals=xvals)
        boots[b] = fit
    return boots.mean(0), np.percentile(boots, 2.5, axis=0), np.percentile(boots, 97.5, axis=0)

mean_fit, lo_band, hi_band = bootstrap_loess_band(d_vals, sq_errs, d_plot_grid)
L_zzb_plot = np.exp(interp(np.log(d_plot_grid), log_d_grid, np.log(np.clip(L_zzb_on_grid, 1e-12, None))))

plt.figure(figsize=(8, 5))
plt.scatter(d_vals[~was_enriched], sq_errs[~was_enriched], s=15, alpha=0.35,
            c='steelblue', label='original draw')
plt.scatter(d_vals[was_enriched], sq_errs[was_enriched], s=20, alpha=0.6,
            c='darkorange', label='enriched (<4m targeted draw)')
plt.plot(d_plot_grid, mean_fit, 'k-', lw=2, label='LOESS fit')
plt.fill_between(d_plot_grid, lo_band, hi_band, color='k', alpha=0.15, label='95% bootstrap band')
plt.plot(d_plot_grid, L_zzb_plot, 'r--', lw=2, label='$L_{ZZB}(d)$')
plt.xscale('log'); plt.yscale('log')
plt.xlabel('separation $d_{true}$ (m)'); plt.ylabel('squared error (m$^2$)')
plt.title('Continuous validation: no bins, enrichment shown explicitly')
plt.legend(fontsize=9)
plt.tight_layout()
plt.savefig('continuous_validation.png', dpi=150)
plt.show()


In [ ]:
# Cell A6 — Continuous falsifiability test (replaces the 4-bin pass/fail table)
# Pass criterion, evaluated pointwise across the observed range:
#   lower bootstrap CI band >= L_ZZB(d)
pass_mask = lo_band >= L_zzb_plot
frac_pass = pass_mask.mean()
print(f"Fraction of the observed d-range where the lower CI band clears L_ZZB(d): {frac_pass:.3f}")

if not pass_mask.all():
    worst_d = d_plot_grid[~pass_mask]
    print(f"Range failing the margin: d in [{worst_d.min():.2f}, {worst_d.max():.2f}] m")
    print("This region needs attention before claiming the floor holds continuously "
          "(check sample density there with Cell A7-style diagnostics, or extend sampling).")
else:
    print("Floor holds across the ENTIRE observed range under the 95% CI band -- "
          "materially stronger than the original 4/4-bin result.")


In [ ]:
# Cell A7 — Enrichment sensitivity: does the conclusion depend on the targeted <4m draw?
mask_no_enrich = ~was_enriched
n_no_enrich = int(mask_no_enrich.sum())
n_low_d_no_enrich = int((d_vals[mask_no_enrich] < 4.0).sum())

mean_fit_ne, lo_ne, hi_ne = bootstrap_loess_band(d_vals[mask_no_enrich], sq_errs[mask_no_enrich], d_plot_grid)

plt.figure(figsize=(8, 5))
plt.plot(d_plot_grid, mean_fit, label=f'with enrichment (n={len(d_vals)})')
plt.fill_between(d_plot_grid, lo_band, hi_band, alpha=0.15)
plt.plot(d_plot_grid, mean_fit_ne, '--', label=f'without enrichment (n={n_no_enrich})')
plt.fill_between(d_plot_grid, lo_ne, hi_ne, alpha=0.15)
plt.plot(d_plot_grid, L_zzb_plot, 'r:', label='$L_{ZZB}(d)$')
plt.xscale('log'); plt.yscale('log')
plt.xlabel('separation $d_{true}$ (m)'); plt.ylabel('squared error (m$^2$)')
plt.legend(fontsize=9); plt.title('Sensitivity to the targeted near-threshold enrichment')
plt.tight_layout()
plt.savefig('enrichment_sensitivity.png', dpi=150)
plt.show()

print(f"Footprints with d_true < 4m WITHOUT enrichment: {n_low_d_no_enrich}")
print("Report this number explicitly in the paper -- if it's small (e.g. <20), "
      "say the sub-4m regime is data-limited without the targeted draw, rather "
      "than letting the enrichment step pass silently inside a bin average.")


In [ ]:
# Cell B1 — Synthetic two-echo model + four estimators
# Synthetic config matches the paper's Appendix H setup (Experiment 3 onward):
# sigma_pulse=1.0, sigma_n=0.2, w1=0.4, w2=0.6, dimensionless pulse-width units
# (NOT the same units as Part A's real-GEDI-meters config).
SYN_SIGMA_PULSE = 1.0
SYN_SIGMA_N = 0.2
SYN_W1, SYN_W2 = 0.4, 0.6
SYN_C_RANGE = (-3.0, 3.0)  # centroid prior support

def synth_waveform(t_grid, c_true, d_true, rng):
    tau1, tau2 = c_true - d_true / 2.0, c_true + d_true / 2.0
    mu = SYN_W1 * pulse(t_grid, tau1, SYN_SIGMA_PULSE) + SYN_W2 * pulse(t_grid, tau2, SYN_SIGMA_PULSE)
    return mu + rng.normal(0, SYN_SIGMA_N, size=t_grid.shape)

def neg_log_lik_synth(params, wf, t_grid):
    c, d = params
    tau1, tau2 = c - d / 2.0, c + d / 2.0
    mu = SYN_W1 * pulse(t_grid, tau1, SYN_SIGMA_PULSE) + SYN_W2 * pulse(t_grid, tau2, SYN_SIGMA_PULSE)
    return 0.5 * np.sum((wf - mu) ** 2) / SYN_SIGMA_N ** 2

def est_local_mle(wf, t_grid, rng):
    # single restart, data-driven init only -- represents a WEAKER local search
    c0 = t_grid[np.argmax(wf)]
    res = minimize(neg_log_lik_synth, x0=[c0, 1.0], args=(wf, t_grid), method='Nelder-Mead')
    return abs(res.x[1])

def est_global_mle(wf, t_grid, rng, n_restarts=6):
    c0_guess = t_grid[np.argmax(wf)]
    best, best_cost = None, np.inf
    for _ in range(n_restarts):
        c0 = c0_guess + rng.normal(0, 0.5)
        d0 = rng.uniform(0.05, 3.0)
        res = minimize(neg_log_lik_synth, x0=[c0, d0], args=(wf, t_grid), method='Nelder-Mead')
        if res.fun < best_cost:
            best_cost, best = res.fun, res.x
    return abs(best[1])

def est_grid_search(wf, t_grid, rng, c_grid=None, d_grid=None):
    if c_grid is None:
        c_grid = np.linspace(SYN_C_RANGE[0], SYN_C_RANGE[1], 25)
    if d_grid is None:
        d_grid = np.linspace(0.05, 3.0, 40)  # coarse -- deliberately resolution-limited
    best_cost, best_d = np.inf, d_grid[0]
    for c in c_grid:
        for d in d_grid:
            cost = neg_log_lik_synth([c, d], wf, t_grid)
            if cost < best_cost:
                best_cost, best_d = cost, d
    return best_d

def est_mmse(wf, t_grid, rng, c_grid=None, d_grid=None):
    # Bayes estimator: posterior mean of d, marginalizing centroid over a
    # uniform prior on SYN_C_RANGE. This is the "near-optimal" baseline --
    # its headroom should shrink toward zero if the ZZB floor is real.
    if c_grid is None:
        c_grid = np.linspace(SYN_C_RANGE[0], SYN_C_RANGE[1], 25)
    if d_grid is None:
        d_grid = np.linspace(0.05, 3.0, 60)
    log_post = np.zeros((len(d_grid), len(c_grid)))
    for i, d in enumerate(d_grid):
        for j, c in enumerate(c_grid):
            log_post[i, j] = -neg_log_lik_synth([c, d], wf, t_grid)
    log_post_d = np.log(np.exp(log_post - log_post.max()).sum(axis=1) + 1e-300) + log_post.max()
    post_d = np.exp(log_post_d - log_post_d.max())
    post_d /= post_d.sum()
    return float(np.sum(post_d * d_grid))

ESTIMATORS_SYN = {
    'local_mle': est_local_mle,
    'global_mle': est_global_mle,
    'grid_search': est_grid_search,
    'mmse': est_mmse,
}
print("Synthetic model + 4 estimators (incl. MMSE near-optimal baseline) defined.")


In [ ]:
# Cell B2 — Continuous synthetic headroom curves for all 4 estimators
# Reduced trial count for interactive runtime; the main pipeline's locked
# setting is n_sim_per_hyp=800 for the ZZB floor itself (kept here) but MMSE/
# grid-search are expensive per-trial (2D grid per waveform), so estimator MSE
# uses fewer trials by default -- bump N_TRIALS_SYN before trusting exact numbers.
N_TRIALS_SYN = 60 if not QUICK else 10
d_sweep_syn = np.geomspace(0.05, 3.0, 12)
syn_t_grid = np.arange(-8, 8, 0.05)
syn_c_grid_zzb = np.linspace(SYN_C_RANGE[0], SYN_C_RANGE[1], 30)
H_GRID_SYN = np.geomspace(0.02, 6.0, 20)
N_SIM_SYN_ZZB = 800 if not QUICK else 60

rng_master = np.random.default_rng(1234)

L_zzb_syn = np.array([
    zzb_floor(d, 0.0, syn_t_grid, SYN_SIGMA_PULSE, SYN_SIGMA_N, SYN_W1, SYN_W2,
              syn_c_grid_zzb, H_GRID_SYN, N_SIM_SYN_ZZB, rng_master)
    for d in d_sweep_syn
])

empirical_mse = {name: [] for name in ESTIMATORS_SYN}
for d in d_sweep_syn:
    errs_by_est = {name: [] for name in ESTIMATORS_SYN}
    for _ in range(N_TRIALS_SYN):
        c_true = rng_master.uniform(*SYN_C_RANGE)
        wf = synth_waveform(syn_t_grid, c_true, d, rng_master)
        for name, fn in ESTIMATORS_SYN.items():
            d_hat = fn(wf, syn_t_grid, rng_master)
            errs_by_est[name].append((d_hat - d) ** 2)
    for name in ESTIMATORS_SYN:
        empirical_mse[name].append(np.mean(errs_by_est[name]))

headroom_syn = {name: np.array(empirical_mse[name]) - L_zzb_syn for name in ESTIMATORS_SYN}

with open('synthetic_headroom_curves.pkl', 'wb') as fh:
    pickle.dump({'d_sweep': d_sweep_syn, 'L_zzb': L_zzb_syn,
                 'empirical_mse': empirical_mse, 'headroom': headroom_syn}, fh)

print("d       " + "  ".join(f"{n:>12s}" for n in ESTIMATORS_SYN))
for i, d in enumerate(d_sweep_syn):
    row = "  ".join(f"{empirical_mse[n][i]:12.4f}" for n in ESTIMATORS_SYN)
    print(f"{d:6.3f}  {row}")


In [ ]:
# Cell B3 — Plot: does MMSE's headroom collapse toward the floor?
plt.figure(figsize=(8, 5))
for name in ESTIMATORS_SYN:
    plt.plot(d_sweep_syn, headroom_syn[name], marker='o', label=name)
plt.axhline(0, color='k', lw=0.5)
plt.xscale('log')
plt.xlabel('separation $d$'); plt.ylabel('headroom $H(d)$ = empirical MSE $-\\ L_{ZZB}(d)$')
plt.title('Synthetic headroom by estimator (MMSE should sit closest to zero)')
plt.legend()
plt.tight_layout()
plt.savefig('synthetic_headroom_by_estimator.png', dpi=150)
plt.show()

mmse_headroom = headroom_syn['mmse']
other_headroom_max = np.maximum.reduce([headroom_syn[n] for n in ESTIMATORS_SYN if n != 'mmse'])
mmse_is_lowest = (mmse_headroom <= other_headroom_max + 1e-9).mean()
print(f"Fraction of d-sweep where MMSE headroom <= every other estimator's headroom: {mmse_is_lowest:.2f}")
if mmse_is_lowest < 0.8:
    print("WARNING: MMSE is not consistently the lowest-headroom estimator. "
          "Before writing this up, check the MMSE grid resolution (d_grid/c_grid) "
          "and N_TRIALS_SYN -- this is more likely an implementation/precision issue "
          "than a genuine floor violation (same lesson as the original ZZB debugging arc).")
else:
    print("MMSE tracks the floor most closely, as expected for a near-optimal estimator -- "
          "supports reading the other estimators' larger headroom as genuinely model-limited.")


In [ ]:
# Cell B4 — MMSE on the REAL-ALS footprints (continuous, no bins)
# Per-footprint posterior mean of d, using each footprint's own fitted c_center
# and t_grid (same real GEDI sigma_pulse/sigma_n as Part A). Expensive per
# footprint (2D grid), so subsample for the interactive run -- widen
# N_MMSE_FOOTPRINTS before trusting exact numbers.
N_MMSE_FOOTPRINTS = min(40, len(all_footprints))
rng_mmse_real = np.random.default_rng(7)
mmse_subset_idx = rng_mmse_real.choice(len(all_footprints), N_MMSE_FOOTPRINTS, replace=False)

def mmse_real_footprint(f, sigma_pulse, sigma_n, d_grid, c_grid):
    if 't_grid' not in f or 'waveform' not in f:
        return None  # needs the synthesized-waveform stage from the main pipeline
    t_grid, wf = f['t_grid'], f['waveform']
    log_post = np.zeros((len(d_grid), len(c_grid)))
    for i, d in enumerate(d_grid):
        for j, c in enumerate(c_grid):
            tau1, tau2 = c - d / 2.0, c + d / 2.0
            mu = pulse(t_grid, tau1, sigma_pulse) + pulse(t_grid, tau2, sigma_pulse)
            log_post[i, j] = -0.5 * np.sum((wf - mu) ** 2) / sigma_n ** 2
    log_post_d = np.log(np.exp(log_post - log_post.max()).sum(axis=1) + 1e-300) + log_post.max()
    post_d = np.exp(log_post_d - log_post_d.max())
    post_d /= post_d.sum()
    return float(np.sum(post_d * d_grid))

mmse_results = []
for idx in mmse_subset_idx:
    f = all_footprints[idx]
    if 't_grid' not in f:
        continue
    c_center_f = f['t_grid'][np.argmax(f['waveform'])]
    d_grid_f = np.linspace(max(0.3, f['d_true'] - 10), f['d_true'] + 10, 40)
    c_grid_f = np.linspace(c_center_f - 15, c_center_f + 15, 20)
    d_hat_mmse = mmse_real_footprint(f, SIGMA_PULSE_M, REAL_SIGMA_N, d_grid_f, c_grid_f)
    if d_hat_mmse is not None:
        mmse_results.append({'d_true': f['d_true'], 'sq_err_mmse': (d_hat_mmse - f['d_true']) ** 2,
                              'sq_err_mle': f['sq_err']})

if not mmse_results:
    print("No footprints had 't_grid'/'waveform' fields in this checkpoint -- "
          "MMSE-on-real-data needs the post-synthesis stage (Cell 6 of the main "
          "pipeline). Skipping; Part B's synthetic result (Cell B2/B3) still stands "
          "on its own as the core ablation for #5.")
else:
    d_mmse = np.array([r['d_true'] for r in mmse_results])
    sq_err_mmse = np.array([r['sq_err_mmse'] for r in mmse_results])
    sq_err_mle = np.array([r['sq_err_mle'] for r in mmse_results])
    print(f"MMSE evaluated on {len(mmse_results)} real footprints.")
    print(f"Mean sq_err  -- MLE (fit_two_echo): {sq_err_mle.mean():.3f}   MMSE: {sq_err_mmse.mean():.3f}")
    print("If MMSE's mean error is not meaningfully lower than the MLE's, re-check "
          "the MMSE d_grid/c_grid resolution before concluding the floor comparison "
          "on real data is uninformative.")


## 10. Notebooks in the zip that are not part of this paper
| file | what it is |
|---|---|
| `Untitled29.ipynb` / `Untitled29 (1).ipynb` (identical) | photon-count scaling d* ∝ N^(−1/4), Poisson channel, global MMSE/ZZB on a lattice — the *next* paper (temporal-lidar / CVPR thread) |
| `Untitled10.ipynb` | measurement-compression / sensor-design study (Fourier vs box projections, CRLB vs learned) — CVPR thread |
| `Untitled23.ipynb`, `Untitled25.ipynb` | CRLB-blow-up threshold calibration of κ; swap-floor sketch. κ is explicitly "future work" in the paper, so left out. (Untitled23 stops on a KeyboardInterrupt before producing κ.) |
| `Untitled2 (1).ipynb` | LROC NAC DTM footprint shapefile download (fails: `IndexError`) — lunar-DTM idea |
| `Untitled5 (2).ipynb` | earlier ZZB prototyping (Bhattacharyya → direct MC). Its final direct-MC numbers are superseded by `Untitled28`; it documents the 0/5 → 5/5 debugging arc quoted in App. F |
| `Untitled4 (1).ipynb` | included (Section 1) |
| `Untitled3`, `Untitled27`, `Untitled8`, `Untitled10 (1)` | chess-PGN codec, "Magic Square RSA" CTF generator, OceanGuard unzip, magic-square search — unrelated |

## 11. Issues found while assembling this file (read before submission)
**Not traceable to any logged run in the zip** — Table 1 rows *roughness r=0.612 / N=7,894* and *geolocation r=1.000*; Table 3 numbers (the run never finished); Fig. 3 / App. D probe; App. A numbers (0.6–0.8 onset, 1–8%); original Fig. 1.
Sections 5–8 regenerate them; the regenerated numbers will not equal the paper's and the paper should be updated from them.

**Reproduced:** Sec. 1 → 2.00 nats (logged 2.0024; QUICK rerun 2.0009). Table 2 at paper settings → 72.8% / 41.7% / 6.6% at d = 0.05 / 0.10 / 0.30 (paper 73.7 / 42.0 / 8.9), I_Schur(0.05)=0.0474 exactly as printed.

**Table 4 (29/30):** the primary locked run passes the 2-SE margin in 24/30 rows. The 29/30 comes from re-running *only* the failing rows at higher precision (4 rows) and treating d=1.689 global-MLE as a "heavy-tail characterisation" rather than a floor test. Table 4 prints the original mean±SE, so by the paper's own criterion that table shows 24/30 (d=1.689 global-MLE has a negative lower bound). Either report 24/30 + the targeted re-checks, or re-run all 30 rows at the higher precision.

**GEDI "3–6×" (Table 1):** the plotted metric is |err| / roughness, and bins are roughness / H_NWCS — the denominator is the binning variable, so low bins are inflated mechanically. Section 8's last cell recomputes raw RMSE per bin; use that.

**ALS Table 6:** (i) one floor per bin at a single representative footprint, with n_sim=150 and a 15-point h-grid, not the "locked" 800 / 40 stated in the paper; (ii) the original <4 m bin (n=16) **failed** the margin (12.11 ± 7.69 vs floor 0.0038) and passed only after enrichment to n=36 — the enrichment is disclosed, the prior failure is not; (iii) 'Cell 13' picks the representative footprint with `abs(d_true - median(errs))` (median of squared errors, not of d_true). The continuous ablation in 9b fixes (i) and (iii).

**Ablation notebook:** the uploaded copy ran on a 60-footprint synthetic stand-in; its MMSE baseline had the lowest headroom in only 75% of the sweep (its own warning). Do not cite either result until re-run (9b does this on the real footprints).

**Pulse convention:** the paper's footnote defines the envelope as exp(−t²/σ²) (so FWHM = 2σ√ln2, κ=1.665, σ=1.3503 m for 15 ns), but every notebook simulates exp(−t²/(2σ²)). With σ=1.3503 that pulse has FWHM 3.18 m, not GEDI's 2.25 m (√2 wider). Fix one or the other and state it.